In [ ]:
# ------------------------------
# 00 | Deposit bootstrap (run once)
# ------------------------------
%load_ext autoreload
%autoreload 2
import os, sys
from pathlib import Path

# The deposit root: two levels above this notebook (code/notebooks/ -> root),
# or wherever SAIL_ROOT points.
ROOT = Path(os.environ.get('SAIL_ROOT', Path.cwd().resolve().parents[1]))
for p in (ROOT / 'code' / 'method', ROOT / 'code' / 'analysis'):
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))
import paths
paths.report()


---

## Simulation of Self-Attention


---

In [ ]:
"""
patch_sweep_multilevel_training_1000x1000.py

Patch-size sweep for HALO per-target hologram generation.

WHAT CHANGED FROM patch500_B1_multilevel_training_1000x1000.py
--------------------------------------------------------------
  1. Loops over PATCH SIZES (run.patch_sizes), not just p=500.
  2. Loops over PHYSICS CONDITIONS (top-level `conditions`), matching the
     schema used by fno_rebuttal.json / attention_ablation.json, instead of
     single pad_factor / apply_sinc / fill_factor fields.
  3. paths.save_base replaces paths.save; outputs go to
        <save_base>/<physics_name>/P<patch>/<image_stem>__<timestamp>/
     so the three axes are separable on disk and nothing collides.
  4. Preflight: every patch size must divide H and W, and the epoch budget is
     printed before anything trains.
  5. The commented-out synthetic dataset generators are removed. They were dead
     in the original (xTrainCPU is the single target image) and there is no
     reason to carry ~150 lines of them through a sweep.

Everything else is byte-for-byte the original protocol: same loss (MSE on
sum-normalised intensity, target upsampled -> clamped -> normalised at padded
resolution), same AdamW at lr 1e-4 with weight_decay 1e-4, same lr *= 0.95
every 1000 epochs, same 10000 epochs, same best-by-train-loss checkpointing,
same checkpoint key names. p=500 / ideal must therefore reproduce 52.30 dB on
alley -- that is the sweep's control and the first thing to check.

COST
----
patch_embed and head are applied per token but SHARE their weights, so their
FLOPs are (H/p)^2 * p^2 * d = H^2 * d, independent of patch size. Only the
encoder grows as the patch shrinks (attention as T^2, feed-forward as T). So
per-epoch cost is roughly flat across the sweep and rises ~30% at p=50: the
sweep costs about N x a single run, not N^2. Parameter count moves the other
way -- 195.7M at p=500 down to 5.2M at p=50 -- because the big linears shrink
while the encoder stays fixed.
"""

from datetime import datetime
import json
import os
import shutil
import time

import numpy as np
import torch
import torch.nn.functional as F
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from PIL import Image
from tqdm import tqdm

from utils import resolve_existing_path
from config_handler import ConfigHandler
from experiment_manager import ExperimentManager
from patching import patchify, unpatchify
from stats_torch import normalize_intensity_sum, save_phase_outputs
from physics import hologram_intensity_from_field
from HALO import HALO

try:
    from skimage.metrics import peak_signal_noise_ratio as _psnr
    from skimage.metrics import structural_similarity as _ssim
except ImportError as e:                      # fail loudly, not silently on loss only
    raise ImportError("scikit-image is required: the sweep is compared on PSNR/SSIM "
                      "against the published 52.30 dB, not on training loss. "
                      "pip install scikit-image") from e


def compute_psnr_ssim(recon_np, target_np):
    """The PSNR/SSIM portion of compute_metrics() in evaluate_methods.py, reproduced
    exactly -- MIN-MAX normalisation of each image independently to [0,1]. This is what
    makes these numbers directly comparable to metrics_summary.json and to the published
    52.30 dB (HALO, ideal, alley, p=500), which is the sweep's control."""
    r = np.asarray(recon_np, dtype=np.float32)
    t = np.asarray(target_np, dtype=np.float32)
    r = r - r.min(); r = r / (r.max() + 1e-12)
    t = t - t.min(); t = t / (t.max() + 1e-12)
    return float(_psnr(t, r, data_range=1.0)), float(_ssim(t, r, data_range=1.0))

# -----------------------------
# Torch device + determinism
# -----------------------------
SEED = 0
torch.manual_seed(SEED)
np.random.seed(SEED)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)
if device == "cuda":
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.benchmark = True

timestamp = datetime.now()


def list_image_files(image_dir, exts=(".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff")):
    files = [os.path.join(image_dir, fn) for fn in sorted(os.listdir(image_dir))
             if fn.lower().endswith(exts)]
    if not files:
        raise ValueError(f"No image files found in: {image_dir}")
    return files


# -----------------------------
# Config
# -----------------------------
config_dir = resolve_existing_path(
    str(paths.CONFIGS / "multilevel"),
)
print("Config path:", config_dir)

config_file = "patch_sweep_training"
config = ConfigHandler.load(config_file, search_paths=[config_dir])
source_config_path = os.path.join(config_dir, config_file + ".json")

save_base = resolve_existing_path(*config.paths.save_base, make=True)
multiple_images_path = resolve_existing_path(*config.paths.multiple_images)

image_files = list_image_files(multiple_images_path)
target_subset = getattr(config.run, "target_subset", None)
if target_subset is not None:
    stems = {os.path.splitext(os.path.basename(f))[0]: f for f in image_files}
    missing = [s for s in target_subset if s not in stems]
    if missing:
        raise ValueError(f"run.target_subset lists images not in {multiple_images_path}: "
                         f"{missing}. Available: {sorted(stems)}")
    image_files = [stems[s] for s in target_subset]
    print(f"SUBSET MODE: {len(image_files)} of {len(stems)} images")

H = config.run.height
W = config.run.width
PATCH_SIZES = list(config.run.patch_sizes)
EPOCHS = config.hyperparameters.epochs
B = config.hyperparameters.batch_size
LR0 = config.hyperparameters.learning_rate
WD = config.hyperparameters.weight_decay
LR_DECAY_EVERY = config.hyperparameters.lr_decay_every
LR_DECAY_FACTOR = config.hyperparameters.lr_decay_factor

A = config.architecture
D_MODEL = A.embedding_dimension
NHEAD = A.heads
NLAYERS = A.layers
FF_DIM = A.feed_forward_dim
DROPOUT = A.dropout
PRE_NORM = bool(A.pre_norm)

CONDITIONS = [{"physics_name": c.physics_name, "pad_factor": c.pad_factor,
               "apply_sinc": bool(c.apply_sinc), "fill_factor": c.fill_factor,
               "fill_is_areal": bool(c.fill_is_areal)} for c in config.conditions]

eps_field, eps_norm = 1e-6, 1e-12

# -----------------------------
# Preflight -- fail before training, not during
# -----------------------------
bad = [p for p in PATCH_SIZES if H % p or W % p]
if bad:
    raise ValueError(f"patch sizes {bad} do not divide ({H},{W}). "
                     f"Valid divisors of {H}: "
                     f"{[p for p in range(20, H + 1) if H % p == 0 and W % p == 0]}")

for f in image_files:
    shape = Image.open(f).convert("L").size
    if shape != (W, H):
        raise ValueError(f"{os.path.basename(f)} is {shape}, expected ({W},{H})")


def param_count(p):
    T = (H // p) * (W // p)
    enc = NLAYERS * ((4 * D_MODEL * D_MODEL + 4 * D_MODEL)
                     + (D_MODEL * FF_DIM + FF_DIM + FF_DIM * D_MODEL + D_MODEL)
                     + 4 * D_MODEL)
    return (p * p * D_MODEL + D_MODEL) + (D_MODEL * 2 * p * p + 2 * p * p) + T * D_MODEL + enc


n_runs = len(CONDITIONS) * len(PATCH_SIZES) * len(image_files)
print(f"\n{'patch':>6} {'tokens':>7} {'bottleneck':>11} {'params':>14} {'vs p=500':>9}")
ref = param_count(500) if 500 in PATCH_SIZES else param_count(PATCH_SIZES[0])
for p in PATCH_SIZES:
    T = (H // p) * (W // p)
    print(f"{p:>6} {T:>7} {T * D_MODEL:>11,} {param_count(p):>14,} {param_count(p) / ref:>8.3f}x")
print(f"\nconditions: {[c['physics_name'] for c in CONDITIONS]}")
print(f"images    : {len(image_files)}")
print(f"runs      : {len(CONDITIONS)} x {len(PATCH_SIZES)} x {len(image_files)} = {n_runs}")
print(f"epochs    : {n_runs} x {EPOCHS} = {n_runs * EPOCHS:,}")
print("per-epoch cost is roughly flat across patch sizes (see module docstring)\n")

sweep_summary = []
t_sweep = time.perf_counter()
run_index = 0

# -----------------------------
# condition -> patch -> image
# -----------------------------
for cond in CONDITIONS:
    pname = cond["physics_name"]
    PAD, SINC = cond["pad_factor"], cond["apply_sinc"]
    FILL, AREAL = cond["fill_factor"], cond["fill_is_areal"]

    for global_p in PATCH_SIZES:
        for image_file in image_files:
            image_stem = os.path.splitext(os.path.basename(image_file))[0]
            run_index += 1

            print(f"\n{'=' * 80}")
            print(f"[run {run_index}/{n_runs}]  {pname} / P{global_p} / {image_stem}")
            if run_index > 1:
                el = time.perf_counter() - t_sweep
                print(f"  elapsed {el / 3600:.2f} hr | ~{el / (run_index - 1) * (n_runs - run_index + 1) / 3600:.2f} hr left")
            print(f"{'=' * 80}\n")

            experiment = ExperimentManager(
                name=image_stem,
                base_dir=os.path.join(save_base, pname, f"P{global_p}"),
                overwrite=config.experiment.overwrite,
            )
            ckpt_dir = os.path.join(experiment.dir, "checkpoints")
            out_dir = os.path.join(experiment.dir, "outputs")
            os.makedirs(ckpt_dir, exist_ok=True)
            os.makedirs(out_dir, exist_ok=True)
            shutil.copy2(source_config_path,
                         os.path.join(experiment.dir, "run_configuration.json"))

            experiment.log(f"Experiment: {config.experiment.description}")
            experiment.log(f"Date: {timestamp}")
            experiment.log(f"Image: {image_file}")
            experiment.log(f"Condition: {pname} (pad_factor={PAD}, apply_sinc={SINC}, "
                           f"fill_factor={FILL}, fill_is_areal={AREAL})")
            experiment.log(f"Patch: {global_p} -> {(H // global_p) * (W // global_p)} tokens, "
                           f"bottleneck {(H // global_p) * (W // global_p) * D_MODEL}")

            # ---- model ----
            torch.manual_seed(SEED)
            np.random.seed(SEED)
            coarse_model = HALO(
                H=H, W=W, p=global_p, in_channels=1, d_model=D_MODEL, nhead=NHEAD,
                num_layers=NLAYERS, dim_feedforward=FF_DIM, dropout=DROPOUT,
                pre_norm=PRE_NORM, output_mode="patch",
            ).to(device)

            n_params = sum(pp.numel() for pp in coarse_model.parameters() if pp.requires_grad)
            if n_params != param_count(global_p):
                raise RuntimeError(f"param_count predicted {param_count(global_p):,} but the "
                                   f"model has {n_params:,} -- fix param_count before trusting "
                                   f"the sweep budget")
            experiment.log(f"Trainable parameters: {n_params:,}")
            print(f"  {n_params:,} parameters")

            # ---- target ----
            X = np.asarray(Image.open(image_file).convert("L"), dtype=np.float32)
            xTrain = torch.from_numpy(X.reshape(1, H, W)).float()
            if device == "cuda":
                xTrain = xTrain.pin_memory()

            plt.figure(figsize=(5, 5))
            plt.imshow(X, cmap="gray"); plt.title(image_stem); plt.axis("off")
            plt.savefig(os.path.join(experiment.dir, "target_image.png"), dpi=180)
            plt.close()

            opt = torch.optim.AdamW(coarse_model.parameters(), lr=LR0,
                                    betas=(0.9, 0.999), weight_decay=WD)

            best_loss, best_epoch, best_state = float("inf"), -1, None
            train_losses = []
            t0 = time.time()

            for epoch in range(EPOCHS):
                if epoch > 0 and epoch % LR_DECAY_EVERY == 0:
                    for pg in opt.param_groups:
                        pg["lr"] *= LR_DECAY_FACTOR
                lr_now = opt.param_groups[0]["lr"]

                coarse_model.train()
                N = xTrain.shape[0]
                perm = torch.randperm(N)
                train_loss, nb = 0.0, 0

                it = tqdm(range(0, N, B), leave=False,
                          desc=f"  ep {epoch}/{EPOCHS} lr={lr_now:.2e}")
                for i in it:
                    xb = xTrain[perm[i:i + B]].to(device, non_blocking=True)

                    y = coarse_model(patchify(xb, global_p))
                    Y = unpatchify(y, H, W, global_p, C=2)

                    I_pred, _ = hologram_intensity_from_field(
                        Y, eps=eps_field, return_field=True, pad_factor=PAD,
                        apply_sinc=SINC, fill_factor=FILL, fill_is_areal=AREAL)
                    Hp, Wp = I_pred.shape[-2], I_pred.shape[-1]
                    I_pred_E = normalize_intensity_sum(I_pred, eps=eps_norm) * (Hp * Wp)

                    xb_up = F.interpolate(xb.unsqueeze(1), size=(Hp, Wp),
                                          mode="bicubic", align_corners=False
                                          ).squeeze(1).clamp_min(0.0)
                    I_tgt_E = normalize_intensity_sum(xb_up, eps=eps_norm) * (Hp * Wp)

                    loss = F.mse_loss(I_pred_E, I_tgt_E, reduction="mean")
                    opt.zero_grad(set_to_none=True)
                    loss.backward()
                    opt.step()

                    train_loss += float(loss.item()); nb += 1
                    it.set_postfix(loss=float(loss.item()))

                train_loss /= max(1, nb)
                train_losses.append(train_loss)
                experiment.log(f"Epoch {epoch} | LR = {lr_now:.10f} | Train Loss = {train_loss:.10f}")

                if train_loss < best_loss:
                    best_loss, best_epoch = train_loss, epoch
                    best_state = {k: v.detach().clone()
                                  for k, v in coarse_model.state_dict().items()}

            training_seconds = time.time() - t0
            if best_state is not None:
                coarse_model.load_state_dict(best_state)
            coarse_model.eval()

            torch.save({"epoch": best_epoch,
                        "coarse_model_state_dict": coarse_model.state_dict(),
                        "best_loss": best_loss,
                        "training_seconds": training_seconds,
                        "patch": global_p,
                        "physics_name": pname},
                       os.path.join(ckpt_dir, "best_model.pt"))

            # ---- final evaluation ----
            with torch.no_grad():
                xb = xTrain[:1].to(device, non_blocking=True)
                Y = unpatchify(coarse_model(patchify(xb, global_p)), H, W, global_p, C=2)
                I_pred, _ = hologram_intensity_from_field(
                    Y, eps=eps_field, return_field=True, pad_factor=PAD,
                    apply_sinc=SINC, fill_factor=FILL, fill_is_areal=AREAL)
                Hp, Wp = I_pred.shape[-2], I_pred.shape[-1]
                I_pred_E = normalize_intensity_sum(I_pred, eps=eps_norm) * (Hp * Wp)
                xb_up = F.interpolate(xb.unsqueeze(1), size=(Hp, Wp), mode="bicubic",
                                      align_corners=False).squeeze(1).clamp_min(0.0)
                I_tgt_E = normalize_intensity_sum(xb_up, eps=eps_norm) * (Hp * Wp)
                final_mse = F.mse_loss(I_pred_E, I_tgt_E).item()
                final_mae = torch.mean(torch.abs(I_pred_E - I_tgt_E)).item()

                # Score the phase-only reconstruction, not the raw network output:
                # the SLM is phase-only, so magnitude is discarded before propagation.
                ph = torch.atan2(Y[0, 1], Y[0, 0])
                Yu = torch.stack([torch.cos(ph), torch.sin(ph)], 0).unsqueeze(0)
                I_u, _ = hologram_intensity_from_field(
                    Yu, eps=eps_field, return_field=True, pad_factor=PAD,
                    apply_sinc=SINC, fill_factor=FILL, fill_is_areal=AREAL)
                I_u = normalize_intensity_sum(I_u, eps=eps_norm) * (Hp * Wp)
            psnr, ssim = compute_psnr_ssim(I_u[0].detach().cpu().numpy(),
                                           I_tgt_E[0].detach().cpu().numpy())
            print(f"  -> {psnr:.2f} dB / {ssim:.4f}")
            experiment.log(f"PSNR: {psnr:.4f} dB | SSIM: {ssim:.4f}")
            if pname == "ideal" and global_p == 500 and image_stem == "alley":
                print(f"  CONTROL CHECK: p=500 ideal alley = {psnr:.2f} dB, "
                      f"published 52.30 -> {'OK' if abs(psnr - 52.30) < 0.5 else 'MISMATCH'}")

            try:
                save_phase_outputs(Y, out_dir=out_dir, prefix="best")
                I_img = I_pred[0].detach().cpu().numpy()
                Image.fromarray((255 * I_img / (I_img.max() + 1e-8)).astype(np.uint8)).save(
                    os.path.join(out_dir, "best_reconstruction.png"))
            except Exception as e:
                print(f"[WARNING] failed to save outputs: {e}")

            plt.figure()
            plt.plot(train_losses); plt.yscale("log")
            plt.xlabel("Epoch"); plt.ylabel("MSE Loss (log)")
            plt.title(f"{image_stem}  {pname} P{global_p}")
            plt.grid(True, which="both", alpha=0.3)
            plt.savefig(os.path.join(experiment.dir, "training_curve.png"), dpi=180)
            plt.close()

            msg = ("=" * 60 + f"\nImage: {image_file}\nCondition: {pname} | Patch: {global_p}\n"
                   f"Parameters: {n_params:,}\nEpochs: {EPOCHS}, Batch: {B}, LR: {LR0:.2e}\n"
                   f"Best loss: {best_loss:.8f} @ epoch {best_epoch}\n"
                   f"Final train loss: {train_losses[-1]:.8f}\n"
                   f"Final eval MSE: {final_mse:.8f} | MAE: {final_mae:.8f}\n"
                   f"PSNR: {psnr:.4f} dB | SSIM: {ssim:.4f}\n"
                   f"Training time: {training_seconds / 60:.1f} min\n" + "=" * 60 + "\n")
            experiment.write_summary(msg)
            experiment.prepend("Summary\r\n" + msg)
            print(msg)

            with open(os.path.join(experiment.dir, "training_metrics.json"), "w") as f:
                json.dump({"image_name": image_stem, "physics_name": pname,
                           "patch": global_p, "tokens": (H // global_p) * (W // global_p),
                           "params": int(n_params), "best_epoch": int(best_epoch),
                           "best_loss": float(best_loss),
                           "final_train_loss": float(train_losses[-1]),
                           "final_eval_mse": float(final_mse),
                           "final_eval_mae": float(final_mae),
                           "psnr": float(psnr), "ssim": float(ssim),
                           "training_seconds": float(training_seconds),
                           "epochs": int(EPOCHS), "batch_size": int(B),
                           "pad_factor": int(PAD), "apply_sinc": bool(SINC),
                           "fill_factor": float(FILL), "fill_is_areal": bool(AREAL)},
                          f, indent=2)

            sweep_summary.append({"physics_name": pname, "patch": global_p,
                                  "image": image_stem, "params": int(n_params),
                                  "best_loss": float(best_loss), "best_epoch": int(best_epoch),
                                  "final_eval_mse": float(final_mse),
                                  "psnr": float(psnr), "ssim": float(ssim),
                                  "training_min": training_seconds / 60,
                                  "experiment_dir": experiment.dir})
            with open(os.path.join(save_base, "patch_sweep_summary.json"), "w") as f:
                json.dump(sweep_summary, f, indent=2)

            del coarse_model, opt, best_state
            torch.cuda.empty_cache()

# -----------------------------
# Ranking
# -----------------------------
print(f"\n{'=' * 80}\nPATCH SWEEP RANKING -- by mean PSNR (higher is better)\n{'=' * 80}")
for pname in [c["physics_name"] for c in CONDITIONS]:
    rows_all = [r for r in sweep_summary if r["physics_name"] == pname]
    if not rows_all:
        continue
    print(f"\n[{pname}]")
    print(f"{'patch':>6} {'tokens':>7} {'params':>13} {'n':>3} {'mean PSNR':>11} "
          f"{'median':>9} {'mean SSIM':>10} {'mean loss':>12} {'min/run':>8}")
    for p in sorted(PATCH_SIZES,
                    key=lambda q: -np.mean([r["psnr"] for r in rows_all if r["patch"] == q] or [-1e9])):
        rows = [r for r in rows_all if r["patch"] == p]
        if not rows:
            continue
        P = [r["psnr"] for r in rows]
        print(f"{p:>6} {(H // p) * (W // p):>7} {rows[0]['params']:>13,} {len(rows):>3} "
              f"{np.mean(P):>11.2f} {np.median(P):>9.2f} "
              f"{np.mean([r['ssim'] for r in rows]):>10.4f} "
              f"{np.mean([r['best_loss'] for r in rows]):>12.3e} "
              f"{np.mean([r['training_min'] for r in rows]):>8.1f}")

    print(f"\n  per-target PSNR")
    stems = sorted({r["image"] for r in rows_all})
    print(f"  {'target':<14}" + "".join(f"{'P' + str(p):>10}" for p in PATCH_SIZES))
    for st in stems:
        line = f"  {st:<14}"
        for p in PATCH_SIZES:
            v = [r["psnr"] for r in rows_all if r["patch"] == p and r["image"] == st]
            line += f"{v[0]:>10.2f}" if v else f"{'--':>10}"
        print(line)

print(f"\ntotal wall clock: {(time.perf_counter() - t_sweep) / 3600:.2f} hr")
print(f"summary: {os.path.join(save_base, 'patch_sweep_summary.json')}")

---

## SAIL

---

In [ ]:
"""
sail_citl_transformer.py

SAIL: per-target CITL transformer training, looped across ALL targets listed
in exposure_settings.json, one target at a time -- each target gets its own
freshly-initialized HALO model, trained from scratch with real camera feedback
via a straight-through estimator (loss_cam uses the real capture, loss_sim
uses the internal simulated prediction).

Structurally adapted from camera_feedback_gs_gd.py (B2) for:
  - rig-wide calibration (auto or manual, matrix-based warpPerspective)
  - per-target exposure (ISO/Tv from exposure_settings.json)
  - padding-aware capture pipeline (out_hw = OUT_HW_LOSS when pad_factor>1)
  - first/best/last capture retention (raw + processed), pruned as-you-go

The training loop itself (model, optimizer, straight-through loss, lambda_sim
decay, LR schedule) is unchanged from the original SAIL notebook cell --
only the surrounding infrastructure (looping, exposure, calibration, pruning,
physics threading) is new.

Depends on: physics.py, HALO.py, patching.py, stats_torch.py, hardware.py,
canon_camera.py, exposure_lookup.py, auto_align.py, and your project's
utils/config_handler/experiment_manager.
"""

from datetime import datetime
import os
import time
import json
import shutil
from pathlib import Path

import numpy as np
import torch
import torch.nn.functional as F
from PIL import Image
import matplotlib.pyplot as plt
import cv2

from canon_camera import CanonEDSDKCamera
from holoeye import slmdisplaysdk

from utils import resolve_existing_path
from config_handler import ConfigHandler
from experiment_manager import ExperimentManager
from hardware import tile_to_slm_centered
from stats_torch import normalize_intensity_sum, save_phase_outputs, phase_to_uint8, field_to_phase
from physics import hologram_intensity_from_field
from patching import patchify, unpatchify
from HALO import HALO
from gd import GradientDescentHologram

from exposure_lookup import iso_to_hex, tv_to_hex


from citl_capture import (
    rename_capture, circular_mask_np, find_dc_center_np,
    normalize_intensity_sum_excluding_mask, load_camera_capture_for_citl,
)



def find_target_image(image_files_dir, stem, exts=(".jpg", ".jpeg", ".png", ".tif", ".tiff", ".bmp")):
    for ext in exts:
        candidate = os.path.join(image_files_dir, stem + ext)
        if os.path.exists(candidate):
            return candidate
    raise FileNotFoundError(
        f"No image file found for stem {stem!r} in {image_files_dir} "
        f"(tried extensions: {exts}). Check exposure_settings.json's target "
        f"keys match the actual filenames exactly (case-sensitive, no extension)."
    )


def save_intensity_png(img_np, out_path, gamma=0.5, percentile=99.0):
    """
    Percentile + gamma display scaling for best_reconstruction.png specifically
    -- confirmed via a real saved checkpoint that plain /max() normalization
    collapses this file to near-black: I_pred's raw stats were max=394.3 vs
    mean=1.0, with only 0.41% of pixels above 1% of the peak, and uint8
    truncation (not rounding) sends everything else to exactly 0. Same
    percentile/gamma approach already used in save_condition_figure
    in earlier work for the same underlying reason. NOT applied to the
    camera-derived saves (best_camera_capture*, processed/*) -- those are
    real, exposure-tuned sensor readouts and were visually confirmed fine as-is.
    """
    img = np.clip(img_np, 0, None)
    vmax = np.percentile(img, percentile)
    if vmax <= 1e-12:
        vmax = float(img.max()) + 1e-12
    disp = np.clip(img / vmax, 0.0, 1.0) ** gamma
    Image.fromarray((255 * disp).astype(np.uint8)).save(out_path)


# -----------------------------
# Config
# -----------------------------
config_dir = resolve_existing_path(
    str(paths.CONFIGS / "citl"),
)
print("Config path:", config_dir)

config_file = "experimental_aberration_sweep"  
config = ConfigHandler.load(config_file, search_paths=[config_dir])

save_base = resolve_existing_path(*config.paths.save_base, make=True)
image_files_dir = resolve_existing_path(*config.paths.image_files)
exposure_settings_path = resolve_existing_path(*config.paths.exposure_settings)

with open(exposure_settings_path) as f:
    exposure = json.load(f)

target_stems = list(exposure["targets"].keys())
target_stems = ["custom"]
print(f"Loaded exposure settings for {len(target_stems)} targets: {target_stems}")

source_config_path = os.path.join(config_dir, config_file + ".json")

# -----------------------------
# Hardware + physics settings (rig-wide / run-wide, shared across every target)
# -----------------------------
SLM_SHAPE = tuple(config.hardware.slm_shape)
DC_RADIUS = config.hardware.dc_radius
DC_AUTO_CENTER = config.hardware.dc_auto_center
DC_CENTER = tuple(config.hardware.dc_center)
SETTLE_TIME_S = config.hardware.settle_time_s
CAPTURE_TIMEOUT_S = config.hardware.capture_timeout_s
DLL_PATH = config.hardware.dll_path

H = config.run.height
W = config.run.width

# Required, not defaulted -- see B2 for why (Axis A/B conflation risk).
LAMBDA_SIM = config.run.lambda_sim

CALIB_BURNIN_ITERATIONS = getattr(config.run, "calib_burnin_iterations", 200)
SAVE_ALL_EPOCH_PNGS = getattr(config.run, "save_all_epoch_pngs", False)

# -----------------------------
# Conditions (ideal/faithful) -- no init-variant axis for SAIL (no
# warm-start concept), so this is just the flat list from config directly.
# Doubles as bookkeeping for what actually ran.
# -----------------------------
run_configs = [
    {
        "physics_name": c.physics_name,
        "pad_factor": c.pad_factor,
        "apply_sinc": c.apply_sinc,
        "fill_factor": c.fill_factor,
        "fill_is_areal": c.fill_is_areal,
    }
    for c in config.conditions
]
print(f"Loaded {len(run_configs)} run configs: " +
      ", ".join(rc["physics_name"] for rc in run_configs))

# -----------------------------
# Hyperparameters
# -----------------------------
lr0 = config.hyperparameters.learning_rate
epochs = config.hyperparameters.epochs
coarse_p = config.hyperparameters.coarse_patch
num_heads = config.hyperparameters.heads
num_layers = config.hyperparameters.layers
feed_forward_dimension = config.hyperparameters.feed_forward_dim
dropout = config.hyperparameters.dropout
pre_norm = config.hyperparameters.pre_norm
d = config.hyperparameters.embedding_dimension

eps_field = 1e-6
eps_norm = 1e-12

make_video = bool(config.experiment.make_video)
frame_rate = int(getattr(config.experiment, "frame_sampling_rate", 20))

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

np.random.seed(0)
torch.manual_seed(0)


def phase_to_slm_frame(phase_np_or_t):
    if isinstance(phase_np_or_t, torch.Tensor):
        phase_np = phase_np_or_t.detach().cpu().numpy().astype(np.float32)
    else:
        phase_np = np.asarray(phase_np_or_t, dtype=np.float32)
    phase_8bit = phase_to_uint8(phase_np)
    holo_small = np.fliplr(phase_8bit).copy()
    holo_slm = tile_to_slm_centered(holo_small, slm_shape=SLM_SHAPE)
    return np.ascontiguousarray(holo_slm.astype(np.uint8))


def load_and_prepare_target(image_path):
    """Load one target and prepare both the (1,H,W) training tensor and the
    display copy. SAIL trains directly in intensity space (no amplitude/
    intensity optimization-domain split like GS/GD -- I_pred and I_tgt_E are
    both intensity throughout), so unlike B2 there is no target_formulation
    branch here."""
    img = Image.open(image_path).convert("L")
    X = np.asarray(img, dtype=np.float32)
    if X.shape != (H, W):
        raise ValueError(f"Target image shape {X.shape} != expected ({H},{W}) for {image_path}")
    return X


try:
    # -----------------------------
    # Open hardware (once, before any target)
    # -----------------------------
    slm = slmdisplaysdk.SLMInstance()
    if not slm.requiresVersion(5):
        raise RuntimeError("Required SDK version not available.")
    error = slm.open()
    if error != slmdisplaysdk.ErrorCode.NoError:
        raise RuntimeError(slm.errorString(error))
    print("SLM opened.")

    camera = CanonEDSDKCamera(
        dll_path=DLL_PATH,
        save_dir=os.path.join(save_base, "_rig_calibration", "dslr captures"),
        auto_set_save_to_host=True,
        auto_set_capacity=True,
        verbose=True,
    )
    camera.initialize()
    camera.open_session()
    print("Camera opened.")

    camera.assert_manual_mode()
    print("Confirmed: camera is in Manual (M) exposure mode.")

    # ========================================================================
    # ONE-TIME, RIG-WIDE ALIGNMENT CALIBRATION (adopted from B2 verbatim)
    # ========================================================================
    rig_calib_dir = os.path.join(save_base, "_rig_calibration")
    os.makedirs(rig_calib_dir, exist_ok=True)

    CALIBRATION_MODE = config.hardware.calibration_mode  # "auto" or "manual"
    print(f"\n=== Alignment calibration (mode={CALIBRATION_MODE}) ===")
    t_calib0 = time.perf_counter()

    if CALIBRATION_MODE == "manual":
        m = config.hardware.manual_alignment
        CAMERA_ROI = {"y0": m.y0, "x0": m.x0, "h": m.h, "w": m.w}
        ROTATE_DEGREES = m.rotation_deg

        calib_json_path = os.path.join(rig_calib_dir, "calibration.json")
        with open(calib_json_path, "w") as f:
            json.dump({
                "mode": "manual",
                "roi": CAMERA_ROI,
                "rotation_deg": ROTATE_DEGREES,
            }, f, indent=2)
        print(
            f"Manual alignment: x0={m.x0}, y0={m.y0}, w={m.w}, h={m.h}, "
            f"rotation_deg={m.rotation_deg}. Saved to: {calib_json_path}"
        )

    else:
        # Auto mode fit a homography via auto_align.calibrate() -- that has
        # no equivalent under the old roi+angle (similarity-transform)
        # calibration this file now uses; a general homography isn't
        # expressible as a simple rect+rotation except in the special case
        # of zero keystone. Rather than silently produce a wrong/mismatched
        # result, fail loudly. Every config in this project has used
        # "manual" mode in practice -- if you genuinely need auto mode,
        # that's new design work, not a mechanical port.
        raise NotImplementedError(
            "CALIBRATION_MODE='auto' is not supported under the old "
            "roi+angle calibration method (citl_capture.py) -- it fit a "
            "homography, which the similarity-transform approach cannot "
            "represent in general. Use CALIBRATION_MODE='manual'."
        )

    t_calib1 = time.perf_counter()
    print(f"Calibration done in {t_calib1 - t_calib0:.1f}s. Transform frozen for the rest of this run "
          f"(rig-wide, independent of physics condition under the old roi+angle calibration method).")

    # ========================================================================
    # OUTER LOOP -- one iteration per physics condition (ideal/faithful).
    # Calibration above is shared across both; everything physics-dependent
    # (pad_factor-derived quantities) is rebuilt fresh here.
    # ========================================================================
    run_summary = []

    for run_config in run_configs:
        PAD_FACTOR = run_config["pad_factor"]
        APPLY_SINC = run_config["apply_sinc"]
        FILL_FACTOR = run_config["fill_factor"]
        FILL_IS_AREAL = run_config["fill_is_areal"]

        M_PAD, N_PAD = H * PAD_FACTOR, W * PAD_FACTOR
        OUT_HW_LOSS = (M_PAD, N_PAD)
        DC_CENTER_PADDED = (DC_CENTER[0] * PAD_FACTOR, DC_CENTER[1] * PAD_FACTOR)
        DC_RADIUS_PADDED = DC_RADIUS * PAD_FACTOR

        save_path = os.path.join(save_base, run_config["physics_name"])
        os.makedirs(save_path, exist_ok=True)

        print(
            f"\n{'=' * 70}\n"
            f"# RUN CONFIG: {run_config['physics_name']} (pad_factor={PAD_FACTOR}) "
            f"-- {len(target_stems)} targets\n"
            f"{'=' * 70}"
        )
        t_run_start = time.perf_counter()

        # -----------------------------
        # PER-TARGET LOOP -- fresh model + optimizer per target, exposure set
        # per target, calibration/hardware shared
        # -----------------------------
        for target_idx, stem in enumerate(target_stems):
            print(f"\n{'#' * 70}\n# TARGET {target_idx + 1}/{len(target_stems)}: {stem}\n{'#' * 70}")

            image_path = find_target_image(image_files_dir, stem)

            experiment = ExperimentManager(
                name=f"{stem}_{run_config['physics_name']}",
                base_dir=save_path,
                overwrite=config.experiment.overwrite,
            )

            camera_dir = os.path.join(experiment.dir, "dslr captures")
            ckpt_dir = os.path.join(experiment.dir, "checkpoints")
            out_dir = os.path.join(experiment.dir, "outputs")
            frames_dir = os.path.join(experiment.dir, "frames")
            processed_dir = os.path.join(experiment.dir, "processed")
            for dir_path in (camera_dir, ckpt_dir, out_dir, frames_dir, processed_dir):
                os.makedirs(dir_path, exist_ok=True)

            camera.save_dir = Path(camera_dir)

            timestamp = datetime.now()
            experiment.log(f"Experiment Details:\r\nExperiment: {config.experiment.description}")
            experiment.log(f"Date: {timestamp}")
            experiment.log(f"Image: {image_path}")
            experiment.log(f"Epochs: {epochs}")
            experiment.log(
                f"Forward model: pad_factor={PAD_FACTOR}, apply_sinc={APPLY_SINC}, "
                f"fill_factor={FILL_FACTOR}, fill_is_areal={FILL_IS_AREAL} "
                f"-> loss/capture resolution ({M_PAD},{N_PAD})"
            )
            if CALIBRATION_MODE == "manual":
                experiment.log(f"Calibration: manual (rig-wide, see {rig_calib_dir}/calibration.json)")
            else:
                experiment.log(f"Calibration: auto (rig-wide, see {rig_calib_dir}/calibration.json)")

            dest_config_path = os.path.join(experiment.dir, "run_configuration.json")
            shutil.copy2(source_config_path, dest_config_path)
            experiment.log(f"Saved config snapshot to: {dest_config_path}")

            # -----------------------------
            # Per-target exposure (ISO + Tv only -- no lens, no Av)
            # -----------------------------
            exp_entry = exposure["targets"][stem]
            iso_val = exp_entry["iso"] if exp_entry["iso"] is not None else exposure["default_iso"]
            tv_val = exp_entry["tv"]
            iso_hex = iso_to_hex(iso_val)
            tv_hex = tv_to_hex(tv_val)
            camera.set_iso(iso_hex)
            camera.set_tv(tv_hex)
            experiment.log(f"Exposure: ISO={iso_val} (0x{iso_hex:08X}), Tv={tv_val} (0x{tv_hex:02X})")

            # -----------------------------
            # Load + prepare this target
            # -----------------------------
            X = load_and_prepare_target(image_path)
            xTrain = torch.from_numpy(X.reshape(1, H, W)).float()
            if device == "cuda":
                xTrain = xTrain.pin_memory()

            plt.figure()
            plt.imshow(X, cmap="gray")
            plt.title(f"Target: {stem}")
            plt.axis("off")
            plt.show()

            # -----------------------------
            # Fresh model + optimizer for this target (SAIL overfits one model
            # per target -- no warm-starting or state carried between targets)
            # -----------------------------
            coarse_model = HALO(
                H=H, W=W, p=coarse_p, in_channels=1, d_model=d, nhead=num_heads,
                num_layers=num_layers, dim_feedforward=feed_forward_dimension,
                dropout=dropout, pre_norm=pre_norm, output_mode="patch",
            ).to(device)

            coarse_params = sum(pp.numel() for pp in coarse_model.parameters() if pp.requires_grad)
            experiment.log(f"Coarse trainable parameters: {coarse_params:,}\r\n")

            opt = torch.optim.AdamW(coarse_model.parameters(), lr=lr0, betas=(0.9, 0.999), weight_decay=1e-4)

            train_losses, train_losses_cam, train_losses_sim = [], [], []
            best_loss = float("inf")
            best_epoch = -1
            best_state = None

            # lambda_start, lambda_end, decay_rate = 0.1, 0.0, 0.01  # TEMPORARY - decay test only

            timing = {"forward": [], "slm": [], "camera": [], "backward": [], "epoch": []}

            # -----------------------------
            # First/best/last capture retention -- adopted from B2's
            # iteration_callback prune pattern, inlined here since SAIL's loop
            # is a plain for-loop (no callback hook like gd.py/gs.py have).
            # Keeps at most 3 raw files and 3 processed PNGs per target
            # regardless of `epochs`, same rationale as B2 (padded-resolution
            # captures are NOT free -- see B2 module docstring).
            # -----------------------------
            raw_paths = {}
            best_raw_state = {"path": None, "epoch": None}
            raw_prune_stats = {"deleted": 0, "kept": 0}

            processed_paths = {}
            best_processed_state = {"path": None, "epoch": None}
            processed_prune_stats = {"deleted": 0, "kept": 0}

            def prune_epoch(epoch, is_best):
                is_first = (epoch == 0)
                is_last = (epoch == epochs - 1)

                raw_path = raw_paths.get(epoch)
                if raw_path is not None:
                    if is_best:
                        old_path, old_epoch = best_raw_state["path"], best_raw_state["epoch"]
                        if old_path is not None and old_epoch != 0 and old_path != raw_path and os.path.exists(old_path):
                            os.remove(old_path)
                            raw_prune_stats["deleted"] += 1
                        best_raw_state["path"], best_raw_state["epoch"] = raw_path, epoch
                        raw_prune_stats["kept"] += 1
                    elif is_first or is_last:
                        raw_prune_stats["kept"] += 1
                    else:
                        if os.path.exists(raw_path):
                            os.remove(raw_path)
                            raw_prune_stats["deleted"] += 1

                if not SAVE_ALL_EPOCH_PNGS:
                    processed_path = processed_paths.get(epoch)
                    if processed_path is not None:
                        if is_best:
                            old_path, old_epoch = best_processed_state["path"], best_processed_state["epoch"]
                            if old_path is not None and old_epoch != 0 and old_path != processed_path and os.path.exists(old_path):
                                os.remove(old_path)
                                processed_prune_stats["deleted"] += 1
                            best_processed_state["path"], best_processed_state["epoch"] = processed_path, epoch
                            processed_prune_stats["kept"] += 1
                        elif is_first or is_last:
                            processed_prune_stats["kept"] += 1
                        else:
                            if os.path.exists(processed_path):
                                os.remove(processed_path)
                                processed_prune_stats["deleted"] += 1

            # -----------------------------
            # Training loop -- unchanged from the original SAIL cell except:
            #   - out_hw is now OUT_HW_LOSS (padding-aware), was fixed (H,W)
            #   - camera captures now via matrix-based load_camera_capture_for_citl
            #   - hologram_intensity_from_field now threads physics kwargs
            #   - I_tgt_E is upsampled to match I_pred when PAD_FACTOR > 1
            #   - raw + processed captures are pruned to first/best/last
            # -----------------------------
            t_start = time.perf_counter()

            for epoch in range(epochs):
                t_iter0 = time.perf_counter()
                if epoch > 0 and epoch % 250 == 0:
                    for pg in opt.param_groups:
                        pg["lr"] *= 0.95
                lr_now = opt.param_groups[0]["lr"]

                coarse_model.train()

                t_fwd0 = time.perf_counter()
                xb = xTrain.to(device, non_blocking=True)  # (1,H,W)

                Xp = patchify(xb, coarse_p)
                y = coarse_model(Xp)
                Y = unpatchify(y, H, W, coarse_p, C=2)

                I_pred, _ = hologram_intensity_from_field(
                    Y, eps=eps_field, return_field=True,
                    pad_factor=PAD_FACTOR, apply_sinc=APPLY_SINC,
                    fill_factor=FILL_FACTOR, fill_is_areal=FILL_IS_AREAL,
                )   # (1, M_PAD, N_PAD)
                I_pred = normalize_intensity_sum(I_pred, eps=eps_norm) * (M_PAD * N_PAD)
                t_fwd1 = time.perf_counter()

                t_slm0 = time.perf_counter()
                phase = field_to_phase(Y)
                holo_slm = phase_to_slm_frame(phase[0])

                with torch.no_grad():
                    error = slm.showData(holo_slm)
                    if error != slmdisplaysdk.ErrorCode.NoError:
                        raise RuntimeError(f"SLM error: {slm.errorString(error)}")
                    time.sleep(SETTLE_TIME_S)
                    t_slm1 = time.perf_counter()

                    t_cam0 = time.perf_counter()
                    result = camera.capture_image(timeout_s=CAPTURE_TIMEOUT_S)
                    t_cam1 = time.perf_counter()

                    renamed_path = rename_capture(result.path, capture_dir=camera_dir, stem=f"epoch_{epoch:04d}")
                    raw_paths[epoch] = renamed_path

                    I_cam, cam_resized_np, dc_mask_t, dc_center_used = load_camera_capture_for_citl(
                        image_path=renamed_path,
                        roi=CAMERA_ROI,
                        angle=ROTATE_DEGREES,
                        out_hw=OUT_HW_LOSS,
                        device=device,
                        eps_norm=eps_norm,
                        dc_radius=DC_RADIUS_PADDED,
                        auto_center=DC_AUTO_CENTER,
                        dc_center=DC_CENTER_PADDED,
                        subtract_min=True,
                        median_ksize=0,
                    )

                    processed_path = os.path.join(processed_dir, f"epoch_{epoch:04d}_camera_processed.png")
                    Image.fromarray(
                        np.clip(cam_resized_np / (cam_resized_np.max() + 1e-8) * 255, 0, 255).astype(np.uint8)
                    ).save(processed_path)
                    processed_paths[epoch] = processed_path

                # ------------------
                # Straight-through proxy (unchanged)
                # ------------------
                t_back0 = time.perf_counter()
                I_tgt_E = normalize_intensity_sum(xb, eps=eps_norm) * (H * W)  # (1,H,W), native

                if PAD_FACTOR > 1:
                    # Same upsampling fix validated for the batched model --
                    # I_pred/I_cam are now at (M_PAD,N_PAD), target must match.
                    I_tgt_E = F.interpolate(
                        I_tgt_E.unsqueeze(1), size=(M_PAD, N_PAD),
                        mode="bicubic", align_corners=False,
                    ).squeeze(1)

                I_pred_for_loss = I_pred
                I_proxy = I_pred_for_loss + (I_cam - I_pred_for_loss).detach()

                weight = (~dc_mask_t).unsqueeze(0).to(dtype=I_cam.dtype)

                loss_cam = ((weight * (I_proxy - I_tgt_E) ** 2).sum() / (weight.sum() + 1e-8))
                loss_sim = ((weight * (I_pred_for_loss - I_tgt_E) ** 2).sum() / (weight.sum() + 1e-8))
                loss = loss_cam + LAMBDA_SIM*loss_sim

                opt.zero_grad(set_to_none=True)
                loss.backward()
                opt.step()
                t_back1 = time.perf_counter()

                train_loss = float(loss.item())
                train_loss_cam = float(loss_cam.item())
                train_loss_sim = float(loss_sim.item())
                train_losses.append(train_loss)
                train_losses_cam.append(train_loss_cam)
                train_losses_sim.append(train_loss_sim)

                experiment.log(
                    f"Epoch {epoch} | LR = {lr_now:.6f} | "
                    f"Train Loss = {train_loss:.6f} | Cam Loss = {train_loss_cam:.6f} | Sim Loss = {train_loss_sim:.6f}"
                )

                # ------------------
                # Best checkpoint (unchanged logic, now at padded resolution)
                # ------------------
                is_best = train_loss_cam < best_loss
                if is_best:
                    best_loss = train_loss_cam
                    best_epoch = epoch
                    best_state = {
                        "epoch": epoch,
                        "coarse_model": {k: v.detach().cpu().clone() for k, v in coarse_model.state_dict().items()}
                    }
                    try:
                        save_phase_outputs(Y, out_dir=out_dir, prefix="best")

                        I_img = I_pred[0].detach().cpu().numpy()
                        save_intensity_png(I_img, os.path.join(out_dir, "best_simulated_reconstruction.png"))

                        I_cam_img = I_cam[0].detach().cpu().numpy()
                        I_cam_img = I_cam_img / (I_cam_img.max() + 1e-8)
                        Image.fromarray((255 * I_cam_img).astype(np.uint8)).save(
                            os.path.join(out_dir, "best_camera_capture.png"))

                        Image.fromarray(
                            np.clip(cam_resized_np / (cam_resized_np.max() + 1e-8) * 255, 0, 255).astype(np.uint8)
                        ).save(os.path.join(out_dir, "best_camera_capture_raw.png"))
                    except Exception as e:
                        print(f"[WARNING] Failed to save best outputs at epoch {epoch}: {e}")

                # First/last capture snapshots (explicit copies, survive pruning)
                if epoch == 0:
                    shutil.copy2(processed_path, os.path.join(out_dir, "first_camera_capture.png"))
                    np.save(os.path.join(out_dir, "first_camera_capture_raw.npy"), cam_resized_np)
                if epoch == epochs - 1:
                    shutil.copy2(processed_path, os.path.join(out_dir, "last_camera_capture.png"))
                    np.save(os.path.join(out_dir, "last_camera_capture_raw.npy"), cam_resized_np)

                prune_epoch(epoch, is_best)

                # ------------------
                # Visualization during training (unchanged)
                # ------------------
                if make_video and (epoch % frame_rate == 0 or epoch == epochs - 1):
                    with torch.no_grad():
                        phase_img = field_to_phase(Y)[0].detach().cpu().numpy()
                        fig, axes = plt.subplots(1, 3, figsize=(14, 4), constrained_layout=True)
                        axes[0].imshow(xb[0].detach().cpu().numpy(), cmap="gray")
                        axes[0].set_title("Target"); axes[0].axis("off")
                        im2 = axes[1].imshow(phase_img, cmap="gray", vmin=-np.pi, vmax=np.pi)
                        axes[1].set_title("Predicted Phase"); axes[1].axis("off")
                        fig.colorbar(im2, ax=axes[1], fraction=0.046, pad=0.04).set_label("Phase (rad)")
                        axes[2].imshow(I_cam[0].detach().cpu().numpy(), cmap="gray")
                        axes[2].set_title("Camera Capture"); axes[2].axis("off")
                        fig.suptitle(f"[{stem}] Epoch {epoch} | Loss={train_loss:.4f}", fontsize=14)
                        fig.savefig(os.path.join(frames_dir, f"epoch_{epoch:05d}.png"), dpi=160, bbox_inches="tight")
                        plt.close(fig)

                t_iter1 = time.perf_counter()
                timing["forward"].append(t_fwd1 - t_fwd0)
                timing["slm"].append(t_slm1 - t_slm0)
                timing["camera"].append(t_cam1 - t_cam0)
                timing["backward"].append(t_back1 - t_back0)
                timing["epoch"].append(t_iter1 - t_iter0)

            # -----------------------------
            # Per-target wrap-up
            # -----------------------------
            experiment.log(
                f"Raw capture pruning: kept {raw_prune_stats['kept']}, deleted {raw_prune_stats['deleted']} "
                f"(first + last + running-best only)"
            )
            if SAVE_ALL_EPOCH_PNGS:
                experiment.log("Processed PNG pruning: disabled, kept every epoch (SAVE_ALL_EPOCH_PNGS=True)")
            else:
                experiment.log(
                    f"Processed PNG pruning: kept {processed_prune_stats['kept']}, "
                    f"deleted {processed_prune_stats['deleted']} (first + last + running-best only)"
                )

            if best_state is not None:
                torch.save({
                    "epoch": best_state["epoch"],
                    "coarse_model_state_dict": best_state["coarse_model"],
                    "best_loss": best_loss,
                }, os.path.join(ckpt_dir, "best_model.pt"))
                experiment.log(f"Saved best model from epoch {best_epoch} with loss {best_loss:.6f}")

            run_elapsed_s = time.perf_counter() - t_start
            avg = {k: float(np.mean(v)) for k, v in timing.items()}
            experiment.log(
                f"Timing | forward={avg['forward']:.3f}s | slm={avg['slm']:.3f}s | "
                f"camera={avg['camera']:.3f}s | backward={avg['backward']:.3f}s | epoch={avg['epoch']:.3f}s"
            )

            msg = "=" * 60 + "\n"
            msg += f"Experiment: {config.experiment.description}\n"
            msg += f"Target: {stem}\n"
            msg += f"Date: {timestamp}\n"
            msg += f"Image: {image_path}\n"
            msg += f"Exposure: ISO={iso_val} (0x{iso_hex:08X}), Tv={tv_val} (0x{tv_hex:02X})\n"
            msg += f"Forward model: pad_factor={PAD_FACTOR}, apply_sinc={APPLY_SINC}, fill_factor={FILL_FACTOR}\n"
            msg += f"Epochs: {epochs}, Base LR: {lr0:.2e}\n"
            msg += f"Total runtime: {run_elapsed_s/60:.1f} min ({run_elapsed_s/3600:.2f} hr)\n"
            msg += f"Best epoch: {best_epoch} (cam loss {best_loss:.6f})\n"
            msg += f"Final train loss: {train_losses[-1]:.6f}\n"
            msg += "=" * 60 + "\n\n"

            experiment.write_summary(msg)
            experiment.prepend("Summary\r\n" + msg)
            print(msg)

            plt.figure()
            plt.plot(train_losses, label="total")
            plt.plot(train_losses_cam, label="cam")
            plt.plot(train_losses_sim, label="sim")
            plt.xlabel("Epoch"); plt.ylabel("MSE Loss")
            plt.title(f"SAIL CITL Training [{stem}]")
            plt.grid(True); plt.legend()
            plt.savefig(os.path.join(experiment.dir, "training_curve.png"), dpi=180)
            plt.close()

        t_run_elapsed_hr = (time.perf_counter() - t_run_start) / 3600
        print(f"\nRun config {run_config['physics_name']} complete: "
              f"{len(target_stems)} targets in {t_run_elapsed_hr:.2f} hr.")
        run_summary.append({
            "physics_name": run_config["physics_name"],
            "targets_completed": len(target_stems),
            "elapsed_hr": t_run_elapsed_hr,
        })

    # ========================================================================
    # BOOKKEEPING -- which conditions actually completed, and how long each
    # took. Printed once at the very end.
    # ========================================================================
    print(f"\n{'=' * 70}\nALL RUN CONFIGS COMPLETE ({len(run_summary)}/{len(run_configs)})\n{'=' * 70}")
    for rs in run_summary:
        print(f"  {rs['physics_name']:10s} -- {rs['targets_completed']} targets, {rs['elapsed_hr']:.2f} hr")
    total_hr = sum(rs["elapsed_hr"] for rs in run_summary)
    print(f"Total wall-clock across all run configs: {total_hr:.2f} hr")

finally:
    try:
        slm.close()
    except Exception as e:
        print("SLM close error (may already be closed):", e)
    try:
        camera.close()
    except Exception as e:
        print("Camera close error (may already be closed):", e)
    print("Closed camera and SLM.")

---

## Batched SAIL - one shared model across all targets

---

Per-target SAIL trains 18 independent models. Batched SAIL trains **one** model
on all 18, testing whether a single set of weights can learn a target-agnostic
intensity→phase mapping under real camera feedback.

**Loop order inverts:** `for epoch: for target:` instead of `for target: for epoch:`.
One optimizer step per target → 13,500 steps. Capture count (and wall-clock) is
unchanged; the camera is the bottleneck, not the model.

**Hyperparameters identical to per-target SAIL** (`lr=1e-4`, `λ_sim=0.1`, 750 epochs)
so weight sharing is the only changed variable. LR decays once per epoch, not per target.

**Two meanings of "best":**
- *Per-target capture* → `targets/{stem}/outputs/`, comparable to per-target SAIL.
- *Shared checkpoint* → lowest **mean** cam loss across all 18 targets in an epoch.
  A checkpoint good for one target and bad for 17 isn't the one to keep.

**Watch `training_curve_per_target.png`.** The untested regime is whether one model
can serve 18 phase outputs without trading them off. Divergent curves - some targets
improving while others degrade - is the failure signature, and the mean curve hides it.

In [ ]:
"""
batched_sail_citl_transformer.py

Batched SAIL: ONE shared HALO model trained jointly across ALL targets listed
in exposure_settings.json, with real camera feedback via the same
straight-through estimator used in per-target SAIL.

Difference from sail_citl_transformer.py (per-target SAIL)
-----------------------------------------------------------
Per-target SAIL:   for target:  for epoch:  <fresh model per target>
Batched SAIL:      for epoch:   for target: <one model, shared>

Total captures are identical (18 targets x 750 epochs = 13,500), so wall-clock
is comparable -- the camera is the bottleneck in both cases, not the model.

The optimizer takes ONE step per target (18 steps per epoch), not one
accumulated step per epoch. Sequential hardware acquisition means each target's
capture is a separate hardware round-trip anyway; stepping per target keeps the
update rule identical to per-target SAIL and only changes what the weights are
shared across.

`best` has two distinct meanings here and they are tracked separately:
  - PER-TARGET best capture: lowest cam loss seen for that target, used for the
    per-target output artefacts (best_camera_capture.png etc). Directly
    comparable to per-target SAIL's outputs.
  - SHARED MODEL best checkpoint: lowest MEAN cam loss across all 18 targets in
    a single epoch. This is the only sensible model-level criterion for a shared
    model -- a checkpoint good for one target but bad for the rest is not the
    model we want to keep.

Depends on: physics.py, HALO.py, patching.py, stats_torch.py, hardware.py,
canon_camera.py, exposure_lookup.py, citl_capture.py, and your project's
utils/config_handler/experiment_manager.
"""

from datetime import datetime
import os
import time
import json
import shutil
from pathlib import Path

import numpy as np
import torch
import torch.nn.functional as F
from PIL import Image
import matplotlib.pyplot as plt

from canon_camera import CanonEDSDKCamera
from holoeye import slmdisplaysdk

from utils import resolve_existing_path
from config_handler import ConfigHandler
from experiment_manager import ExperimentManager
from hardware import tile_to_slm_centered
from stats_torch import normalize_intensity_sum, save_phase_outputs, phase_to_uint8, field_to_phase
from physics import hologram_intensity_from_field
from patching import patchify, unpatchify
from HALO import HALO

from exposure_lookup import iso_to_hex, tv_to_hex

from citl_capture import (
    rename_capture, circular_mask_np, find_dc_center_np,
    normalize_intensity_sum_excluding_mask, load_camera_capture_for_citl,
)


def find_target_image(image_files_dir, stem, exts=(".jpg", ".jpeg", ".png", ".tif", ".tiff", ".bmp")):
    for ext in exts:
        candidate = os.path.join(image_files_dir, stem + ext)
        if os.path.exists(candidate):
            return candidate
    raise FileNotFoundError(
        f"No image file found for stem {stem!r} in {image_files_dir} "
        f"(tried extensions: {exts}). Check exposure_settings.json's target "
        f"keys match the actual filenames exactly (case-sensitive, no extension)."
    )


def save_intensity_png(img_np, out_path, gamma=0.5, percentile=99.0):
    """Percentile + gamma display scaling for the simulated reconstruction only
    -- see per-target SAIL for the full rationale (plain /max() collapses this
    file to near-black). NOT applied to camera-derived saves."""
    img = np.clip(img_np, 0, None)
    vmax = np.percentile(img, percentile)
    if vmax <= 1e-12:
        vmax = float(img.max()) + 1e-12
    disp = np.clip(img / vmax, 0.0, 1.0) ** gamma
    Image.fromarray((255 * disp).astype(np.uint8)).save(out_path)


# -----------------------------
# Config
# -----------------------------
config_dir = resolve_existing_path(
    str(paths.CONFIGS / "citl"),
)
print("Config path:", config_dir)

config_file = "batched_sail"
config = ConfigHandler.load(config_file, search_paths=[config_dir])

save_base = resolve_existing_path(*config.paths.save_base, make=True)
image_files_dir = resolve_existing_path(*config.paths.image_files)
exposure_settings_path = resolve_existing_path(*config.paths.exposure_settings)

with open(exposure_settings_path) as f:
    exposure = json.load(f)

target_stems = list(exposure["targets"].keys())
print(f"Loaded exposure settings for {len(target_stems)} targets: {target_stems}")

source_config_path = os.path.join(config_dir, config_file + ".json")

# -----------------------------
# Hardware + run settings (rig-wide, shared across every target)
# -----------------------------
SLM_SHAPE = tuple(config.hardware.slm_shape)
DC_RADIUS = config.hardware.dc_radius
DC_AUTO_CENTER = config.hardware.dc_auto_center
DC_CENTER = tuple(config.hardware.dc_center)
SETTLE_TIME_S = config.hardware.settle_time_s
CAPTURE_TIMEOUT_S = config.hardware.capture_timeout_s
DLL_PATH = config.hardware.dll_path

H = config.run.height
W = config.run.width

LAMBDA_SIM = config.run.lambda_sim
SAVE_ALL_EPOCH_PNGS = getattr(config.run, "save_all_epoch_pngs", False)

run_configs = [
    {
        "physics_name": c.physics_name,
        "pad_factor": c.pad_factor,
        "apply_sinc": c.apply_sinc,
        "fill_factor": c.fill_factor,
        "fill_is_areal": c.fill_is_areal,
    }
    for c in config.conditions
]
print(f"Loaded {len(run_configs)} run configs: " +
      ", ".join(rc["physics_name"] for rc in run_configs))

# -----------------------------
# Hyperparameters
# -----------------------------
lr0 = config.hyperparameters.learning_rate
epochs = config.hyperparameters.epochs
coarse_p = config.hyperparameters.coarse_patch
num_heads = config.hyperparameters.heads
num_layers = config.hyperparameters.layers
feed_forward_dimension = config.hyperparameters.feed_forward_dim
dropout = config.hyperparameters.dropout
pre_norm = config.hyperparameters.pre_norm
d = config.hyperparameters.embedding_dimension

eps_field = 1e-6
eps_norm = 1e-12

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

np.random.seed(0)
torch.manual_seed(0)


def phase_to_slm_frame(phase_np_or_t):
    if isinstance(phase_np_or_t, torch.Tensor):
        phase_np = phase_np_or_t.detach().cpu().numpy().astype(np.float32)
    else:
        phase_np = np.asarray(phase_np_or_t, dtype=np.float32)
    phase_8bit = phase_to_uint8(phase_np)
    holo_small = np.fliplr(phase_8bit).copy()
    holo_slm = tile_to_slm_centered(holo_small, slm_shape=SLM_SHAPE)
    return np.ascontiguousarray(holo_slm.astype(np.uint8))


def load_and_prepare_target(image_path):
    img = Image.open(image_path).convert("L")
    X = np.asarray(img, dtype=np.float32)
    if X.shape != (H, W):
        raise ValueError(f"Target image shape {X.shape} != expected ({H},{W}) for {image_path}")
    return X


try:
    # -----------------------------
    # Open hardware (once)
    # -----------------------------
    slm = slmdisplaysdk.SLMInstance()
    if not slm.requiresVersion(5):
        raise RuntimeError("Required SDK version not available.")
    error = slm.open()
    if error != slmdisplaysdk.ErrorCode.NoError:
        raise RuntimeError(slm.errorString(error))
    print("SLM opened.")

    camera = CanonEDSDKCamera(
        dll_path=DLL_PATH,
        save_dir=os.path.join(save_base, "_rig_calibration", "dslr captures"),
        auto_set_save_to_host=True,
        auto_set_capacity=True,
        verbose=True,
    )
    camera.initialize()
    camera.open_session()
    print("Camera opened.")

    camera.assert_manual_mode()
    print("Confirmed: camera is in Manual (M) exposure mode.")

    # ========================================================================
    # ONE-TIME, RIG-WIDE ALIGNMENT CALIBRATION (identical to per-target SAIL)
    # ========================================================================
    rig_calib_dir = os.path.join(save_base, "_rig_calibration")
    os.makedirs(rig_calib_dir, exist_ok=True)

    CALIBRATION_MODE = config.hardware.calibration_mode
    print(f"\n=== Alignment calibration (mode={CALIBRATION_MODE}) ===")

    if CALIBRATION_MODE == "manual":
        m = config.hardware.manual_alignment
        CAMERA_ROI = {"y0": m.y0, "x0": m.x0, "h": m.h, "w": m.w}
        ROTATE_DEGREES = m.rotation_deg

        calib_json_path = os.path.join(rig_calib_dir, "calibration.json")
        with open(calib_json_path, "w") as f:
            json.dump({"mode": "manual", "roi": CAMERA_ROI, "rotation_deg": ROTATE_DEGREES}, f, indent=2)
        print(
            f"Manual alignment: x0={m.x0}, y0={m.y0}, w={m.w}, h={m.h}, "
            f"rotation_deg={m.rotation_deg}. Saved to: {calib_json_path}"
        )
    else:
        raise NotImplementedError(
            "CALIBRATION_MODE='auto' is not supported under the old "
            "roi+angle calibration method (citl_capture.py). Use 'manual'."
        )

    # ========================================================================
    # OUTER LOOP -- one iteration per physics condition (ideal/faithful).
    # ONE shared model per condition, trained across all targets.
    # ========================================================================
    run_summary = []

    for run_config in run_configs:
        PAD_FACTOR = run_config["pad_factor"]
        APPLY_SINC = run_config["apply_sinc"]
        FILL_FACTOR = run_config["fill_factor"]
        FILL_IS_AREAL = run_config["fill_is_areal"]

        M_PAD, N_PAD = H * PAD_FACTOR, W * PAD_FACTOR
        OUT_HW_LOSS = (M_PAD, N_PAD)
        DC_CENTER_PADDED = (DC_CENTER[0] * PAD_FACTOR, DC_CENTER[1] * PAD_FACTOR)
        DC_RADIUS_PADDED = DC_RADIUS * PAD_FACTOR

        save_path = os.path.join(save_base, run_config["physics_name"])
        os.makedirs(save_path, exist_ok=True)

        print(
            f"\n{'=' * 70}\n"
            f"# RUN CONFIG: {run_config['physics_name']} (pad_factor={PAD_FACTOR}) "
            f"-- ONE shared model over {len(target_stems)} targets x {epochs} epochs\n"
            f"{'=' * 70}"
        )
        t_run_start = time.perf_counter()

        # ------------------------------------------------------------------
        # ONE experiment for the whole condition (not one per target) --
        # a shared model is a single training run, so it gets a single log,
        # a single training curve, and a single checkpoint. Per-target
        # artefacts live in per-target subdirectories underneath.
        # ------------------------------------------------------------------
        experiment = ExperimentManager(
            name=f"batched_{run_config['physics_name']}",
            base_dir=save_path,
            overwrite=config.experiment.overwrite,
        )

        ckpt_dir = os.path.join(experiment.dir, "checkpoints")
        os.makedirs(ckpt_dir, exist_ok=True)

        timestamp = datetime.now()
        experiment.log(f"Experiment Details:\r\nExperiment: {config.experiment.description}")
        experiment.log(f"Date: {timestamp}")
        experiment.log(f"Targets ({len(target_stems)}): {target_stems}")
        experiment.log(f"Epochs: {epochs} (one optimizer step per target => {epochs * len(target_stems)} steps)")
        experiment.log(
            f"Forward model: pad_factor={PAD_FACTOR}, apply_sinc={APPLY_SINC}, "
            f"fill_factor={FILL_FACTOR}, fill_is_areal={FILL_IS_AREAL} "
            f"-> loss/capture resolution ({M_PAD},{N_PAD})"
        )
        experiment.log(f"Calibration: manual (rig-wide, see {rig_calib_dir}/calibration.json)")

        dest_config_path = os.path.join(experiment.dir, "run_configuration.json")
        shutil.copy2(source_config_path, dest_config_path)
        experiment.log(f"Saved config snapshot to: {dest_config_path}")

        # ------------------------------------------------------------------
        # Pre-load every target once, up front. 18 x 1000 x 1000 float32 is
        # ~72 MB on device -- cheap, and avoids re-reading from disk 13,500
        # times over the run.
        # ------------------------------------------------------------------
        targets = {}
        per_target_dirs = {}
        for stem in target_stems:
            image_path = find_target_image(image_files_dir, stem)
            X = load_and_prepare_target(image_path)
            targets[stem] = {
                "image_path": image_path,
                "xTrain": torch.from_numpy(X.reshape(1, H, W)).float().to(device),
            }

            target_root = os.path.join(experiment.dir, "targets", stem)
            dirs = {
                "camera": os.path.join(target_root, "dslr captures"),
                "out": os.path.join(target_root, "outputs"),
                "processed": os.path.join(target_root, "processed"),
            }
            for dir_path in dirs.values():
                os.makedirs(dir_path, exist_ok=True)
            per_target_dirs[stem] = dirs

            exp_entry = exposure["targets"][stem]
            iso_val = exp_entry["iso"] if exp_entry["iso"] is not None else exposure["default_iso"]
            targets[stem]["iso_val"] = iso_val
            targets[stem]["tv_val"] = exp_entry["tv"]
            targets[stem]["iso_hex"] = iso_to_hex(iso_val)
            targets[stem]["tv_hex"] = tv_to_hex(exp_entry["tv"])
            experiment.log(
                f"[{stem}] image={image_path} | Exposure: ISO={iso_val} "
                f"(0x{targets[stem]['iso_hex']:08X}), Tv={exp_entry['tv']} (0x{targets[stem]['tv_hex']:02X})"
            )

        # ------------------------------------------------------------------
        # ONE model + ONE optimizer for the whole condition. This is the
        # entire structural difference from per-target SAIL -- everything
        # below is the same training step, just with shared weights.
        # ------------------------------------------------------------------
        coarse_model = HALO(
            H=H, W=W, p=coarse_p, in_channels=1, d_model=d, nhead=num_heads,
            num_layers=num_layers, dim_feedforward=feed_forward_dimension,
            dropout=dropout, pre_norm=pre_norm, output_mode="patch",
        ).to(device)

        coarse_params = sum(pp.numel() for pp in coarse_model.parameters() if pp.requires_grad)
        experiment.log(f"Shared model trainable parameters: {coarse_params:,}\r\n")

        opt = torch.optim.AdamW(coarse_model.parameters(), lr=lr0, betas=(0.9, 0.999), weight_decay=1e-4)

        # Per-target best tracking (comparable to per-target SAIL's outputs)
        best_loss = {stem: float("inf") for stem in target_stems}
        best_epoch = {stem: -1 for stem in target_stems}

        # Shared-model best checkpoint, selected on MEAN cam loss across targets
        best_mean_cam = float("inf")
        best_mean_epoch = -1
        best_state = None

        # Per-epoch mean curves (one point per epoch, averaged over 18 targets)
        epoch_mean_total, epoch_mean_cam, epoch_mean_sim = [], [], []
        # Per-target curves, for the per-target breakdown figure
        per_target_cam = {stem: [] for stem in target_stems}

        # Capture retention: first/best/last per TARGET, keyed (stem, epoch)
        raw_paths = {}
        processed_paths = {}
        best_raw_state = {stem: {"path": None, "epoch": None} for stem in target_stems}
        best_processed_state = {stem: {"path": None, "epoch": None} for stem in target_stems}
        raw_prune_stats = {"deleted": 0, "kept": 0}
        processed_prune_stats = {"deleted": 0, "kept": 0}

        def prune_epoch(stem, epoch, is_best):
            is_first = (epoch == 0)
            is_last = (epoch == epochs - 1)

            raw_path = raw_paths.get((stem, epoch))
            if raw_path is not None:
                if is_best:
                    old = best_raw_state[stem]
                    if (old["path"] is not None and old["epoch"] != 0
                            and old["path"] != raw_path and os.path.exists(old["path"])):
                        os.remove(old["path"])
                        raw_prune_stats["deleted"] += 1
                    best_raw_state[stem] = {"path": raw_path, "epoch": epoch}
                    raw_prune_stats["kept"] += 1
                elif is_first or is_last:
                    raw_prune_stats["kept"] += 1
                else:
                    if os.path.exists(raw_path):
                        os.remove(raw_path)
                        raw_prune_stats["deleted"] += 1
                raw_paths.pop((stem, epoch), None)

            if not SAVE_ALL_EPOCH_PNGS:
                processed_path = processed_paths.get((stem, epoch))
                if processed_path is not None:
                    if is_best:
                        old = best_processed_state[stem]
                        if (old["path"] is not None and old["epoch"] != 0
                                and old["path"] != processed_path and os.path.exists(old["path"])):
                            os.remove(old["path"])
                            processed_prune_stats["deleted"] += 1
                        best_processed_state[stem] = {"path": processed_path, "epoch": epoch}
                        processed_prune_stats["kept"] += 1
                    elif is_first or is_last:
                        processed_prune_stats["kept"] += 1
                    else:
                        if os.path.exists(processed_path):
                            os.remove(processed_path)
                            processed_prune_stats["deleted"] += 1
                    processed_paths.pop((stem, epoch), None)

        timing = {"forward": [], "slm": [], "camera": [], "backward": [], "step": []}

        # ==================================================================
        # TRAINING -- epoch is now the OUTER loop, target the INNER loop.
        # One optimizer step per (epoch, target) pair.
        # ==================================================================
        t_start = time.perf_counter()

        for epoch in range(epochs):
            # LR schedule steps once per EPOCH (not once per target), so the
            # decay lands at the same wall-clock points as per-target SAIL.
            if epoch > 0 and epoch % 250 == 0:
                for pg in opt.param_groups:
                    pg["lr"] *= 0.95
            lr_now = opt.param_groups[0]["lr"]

            coarse_model.train()

            epoch_losses_total, epoch_losses_cam, epoch_losses_sim = [], [], []

            for stem in target_stems:
                t_step0 = time.perf_counter()
                tgt = targets[stem]
                dirs = per_target_dirs[stem]

                # Exposure is per-target, so it must be set on every inner
                # iteration -- unlike per-target SAIL where it was set once.
                camera.set_iso(tgt["iso_hex"])
                camera.set_tv(tgt["tv_hex"])
                camera.save_dir = Path(dirs["camera"])

                t_fwd0 = time.perf_counter()
                xb = tgt["xTrain"]  # (1,H,W), already on device

                Xp = patchify(xb, coarse_p)
                y = coarse_model(Xp)
                Y = unpatchify(y, H, W, coarse_p, C=2)

                I_pred, _ = hologram_intensity_from_field(
                    Y, eps=eps_field, return_field=True,
                    pad_factor=PAD_FACTOR, apply_sinc=APPLY_SINC,
                    fill_factor=FILL_FACTOR, fill_is_areal=FILL_IS_AREAL,
                )
                I_pred = normalize_intensity_sum(I_pred, eps=eps_norm) * (M_PAD * N_PAD)
                t_fwd1 = time.perf_counter()

                t_slm0 = time.perf_counter()
                phase = field_to_phase(Y)
                holo_slm = phase_to_slm_frame(phase[0])

                with torch.no_grad():
                    error = slm.showData(holo_slm)
                    if error != slmdisplaysdk.ErrorCode.NoError:
                        raise RuntimeError(f"SLM error: {slm.errorString(error)}")
                    time.sleep(SETTLE_TIME_S)
                    t_slm1 = time.perf_counter()

                    t_cam0 = time.perf_counter()
                    result = camera.capture_image(timeout_s=CAPTURE_TIMEOUT_S)
                    t_cam1 = time.perf_counter()
                    time.sleep(0.5)  # let EDSDK finish the capture/transfer before next property write

                    renamed_path = rename_capture(
                        result.path, capture_dir=dirs["camera"], stem=f"epoch_{epoch:04d}"
                    )
                    raw_paths[(stem, epoch)] = renamed_path

                    I_cam, cam_resized_np, dc_mask_t, dc_center_used = load_camera_capture_for_citl(
                        image_path=renamed_path,
                        roi=CAMERA_ROI,
                        angle=ROTATE_DEGREES,
                        out_hw=OUT_HW_LOSS,
                        device=device,
                        eps_norm=eps_norm,
                        dc_radius=DC_RADIUS_PADDED,
                        auto_center=DC_AUTO_CENTER,
                        dc_center=DC_CENTER_PADDED,
                        subtract_min=True,
                        median_ksize=0,
                    )

                    processed_path = os.path.join(
                        dirs["processed"], f"epoch_{epoch:04d}_camera_processed.png"
                    )
                    Image.fromarray(
                        np.clip(cam_resized_np / (cam_resized_np.max() + 1e-8) * 255, 0, 255).astype(np.uint8)
                    ).save(processed_path)
                    processed_paths[(stem, epoch)] = processed_path

                # ------------------
                # Straight-through proxy (identical to per-target SAIL)
                # ------------------
                t_back0 = time.perf_counter()
                I_tgt_E = normalize_intensity_sum(xb, eps=eps_norm) * (H * W)

                if PAD_FACTOR > 1:
                    I_tgt_E = F.interpolate(
                        I_tgt_E.unsqueeze(1), size=(M_PAD, N_PAD),
                        mode="bicubic", align_corners=False,
                    ).squeeze(1)

                I_pred_for_loss = I_pred
                I_proxy = I_pred_for_loss + (I_cam - I_pred_for_loss).detach()

                weight = (~dc_mask_t).unsqueeze(0).to(dtype=I_cam.dtype)

                loss_cam = ((weight * (I_proxy - I_tgt_E) ** 2).sum() / (weight.sum() + 1e-8))
                loss_sim = ((weight * (I_pred_for_loss - I_tgt_E) ** 2).sum() / (weight.sum() + 1e-8))
                loss = loss_cam + LAMBDA_SIM * loss_sim

                opt.zero_grad(set_to_none=True)
                loss.backward()
                opt.step()
                t_back1 = time.perf_counter()

                train_loss = float(loss.item())
                train_loss_cam = float(loss_cam.item())
                train_loss_sim = float(loss_sim.item())

                epoch_losses_total.append(train_loss)
                epoch_losses_cam.append(train_loss_cam)
                epoch_losses_sim.append(train_loss_sim)
                per_target_cam[stem].append(train_loss_cam)

                experiment.log(
                    f"Epoch {epoch} | {stem} | LR = {lr_now:.6f} | "
                    f"Train Loss = {train_loss:.6f} | Cam Loss = {train_loss_cam:.6f} | "
                    f"Sim Loss = {train_loss_sim:.6f}"
                )

                # ------------------
                # PER-TARGET best capture (not the model checkpoint)
                # ------------------
                is_best = train_loss_cam < best_loss[stem]
                if is_best:
                    best_loss[stem] = train_loss_cam
                    best_epoch[stem] = epoch
                    try:
                        save_phase_outputs(Y, out_dir=dirs["out"], prefix="best")

                        I_img = I_pred[0].detach().cpu().numpy()
                        save_intensity_png(
                            I_img, os.path.join(dirs["out"], "best_simulated_reconstruction.png")
                        )

                        I_cam_img = I_cam[0].detach().cpu().numpy()
                        I_cam_img = I_cam_img / (I_cam_img.max() + 1e-8)
                        Image.fromarray((255 * I_cam_img).astype(np.uint8)).save(
                            os.path.join(dirs["out"], "best_camera_capture.png"))

                        Image.fromarray(
                            np.clip(cam_resized_np / (cam_resized_np.max() + 1e-8) * 255, 0, 255).astype(np.uint8)
                        ).save(os.path.join(dirs["out"], "best_camera_capture_raw.png"))
                    except Exception as e:
                        print(f"[WARNING] Failed to save best outputs for {stem} at epoch {epoch}: {e}")

                if epoch == 0:
                    shutil.copy2(processed_path, os.path.join(dirs["out"], "first_camera_capture.png"))
                    np.save(os.path.join(dirs["out"], "first_camera_capture_raw.npy"), cam_resized_np)
                if epoch == epochs - 1:
                    shutil.copy2(processed_path, os.path.join(dirs["out"], "last_camera_capture.png"))
                    np.save(os.path.join(dirs["out"], "last_camera_capture_raw.npy"), cam_resized_np)

                prune_epoch(stem, epoch, is_best)

                t_step1 = time.perf_counter()
                timing["forward"].append(t_fwd1 - t_fwd0)
                timing["slm"].append(t_slm1 - t_slm0)
                timing["camera"].append(t_cam1 - t_cam0)
                timing["backward"].append(t_back1 - t_back0)
                timing["step"].append(t_step1 - t_step0)

            # --------------------------------------------------------------
            # End of epoch -- shared-model checkpoint on MEAN cam loss.
            # --------------------------------------------------------------
            mean_total = float(np.mean(epoch_losses_total))
            mean_cam = float(np.mean(epoch_losses_cam))
            mean_sim = float(np.mean(epoch_losses_sim))
            epoch_mean_total.append(mean_total)
            epoch_mean_cam.append(mean_cam)
            epoch_mean_sim.append(mean_sim)

            experiment.log(
                f"Epoch {epoch} MEAN over {len(target_stems)} targets | "
                f"Train Loss = {mean_total:.6f} | Cam Loss = {mean_cam:.6f} | Sim Loss = {mean_sim:.6f}"
            )

            if mean_cam < best_mean_cam:
                best_mean_cam = mean_cam
                best_mean_epoch = epoch
                best_state = {
                    "epoch": epoch,
                    "coarse_model": {k: v.detach().cpu().clone()
                                     for k, v in coarse_model.state_dict().items()},
                }

            if epoch % 10 == 0 or epoch == epochs - 1:
                elapsed_hr = (time.perf_counter() - t_start) / 3600
                print(
                    f"[{run_config['physics_name']}] epoch {epoch}/{epochs - 1} | "
                    f"mean cam = {mean_cam:.6f} | best mean cam = {best_mean_cam:.6f} "
                    f"(epoch {best_mean_epoch}) | elapsed {elapsed_hr:.2f} hr"
                )

        # ------------------------------------------------------------------
        # Condition wrap-up
        # ------------------------------------------------------------------
        experiment.log(
            f"Raw capture pruning: kept {raw_prune_stats['kept']}, deleted {raw_prune_stats['deleted']} "
            f"(first + last + running-best, per target)"
        )
        if SAVE_ALL_EPOCH_PNGS:
            experiment.log("Processed PNG pruning: disabled (SAVE_ALL_EPOCH_PNGS=True)")
        else:
            experiment.log(
                f"Processed PNG pruning: kept {processed_prune_stats['kept']}, "
                f"deleted {processed_prune_stats['deleted']} (first + last + running-best, per target)"
            )

        if best_state is not None:
            torch.save({
                "epoch": best_state["epoch"],
                "coarse_model_state_dict": best_state["coarse_model"],
                "best_mean_cam_loss": best_mean_cam,
                "target_stems": target_stems,
            }, os.path.join(ckpt_dir, "best_shared_model.pt"))
            experiment.log(
                f"Saved best SHARED model from epoch {best_mean_epoch} "
                f"with mean cam loss {best_mean_cam:.6f}"
            )

        torch.save({
            "epoch": epochs - 1,
            "coarse_model_state_dict": {k: v.detach().cpu().clone()
                                        for k, v in coarse_model.state_dict().items()},
            "target_stems": target_stems,
        }, os.path.join(ckpt_dir, "last_shared_model.pt"))

        run_elapsed_s = time.perf_counter() - t_start
        avg = {k: float(np.mean(v)) for k, v in timing.items()}
        experiment.log(
            f"Timing | forward={avg['forward']:.3f}s | slm={avg['slm']:.3f}s | "
            f"camera={avg['camera']:.3f}s | backward={avg['backward']:.3f}s | step={avg['step']:.3f}s"
        )

        experiment.log("\n=== Per-target best cam loss (shared model) ===")
        for stem in target_stems:
            experiment.log(f"  {stem:<16} best cam loss = {best_loss[stem]:.6f} (epoch {best_epoch[stem]})")

        msg = "=" * 60 + "\n"
        msg += f"Experiment: {config.experiment.description}\n"
        msg += f"Condition: {run_config['physics_name']} (pad_factor={PAD_FACTOR})\n"
        msg += f"Date: {timestamp}\n"
        msg += f"Targets: {len(target_stems)} ({', '.join(target_stems)})\n"
        msg += f"Epochs: {epochs}, optimizer steps: {epochs * len(target_stems)}, Base LR: {lr0:.2e}\n"
        msg += f"Total runtime: {run_elapsed_s/60:.1f} min ({run_elapsed_s/3600:.2f} hr)\n"
        msg += f"Best shared-model epoch: {best_mean_epoch} (mean cam loss {best_mean_cam:.6f})\n"
        msg += f"Final epoch mean cam loss: {epoch_mean_cam[-1]:.6f}\n"
        msg += "=" * 60 + "\n\n"

        experiment.write_summary(msg)
        experiment.prepend("Summary\r\n" + msg)
        print(msg)

        # Mean training curve
        plt.figure()
        plt.plot(epoch_mean_total, label="total (mean)")
        plt.plot(epoch_mean_cam, label="cam (mean)")
        plt.plot(epoch_mean_sim, label="sim (mean)")
        plt.xlabel("Epoch"); plt.ylabel("MSE Loss")
        plt.title(f"Batched SAIL [{run_config['physics_name']}] -- mean over {len(target_stems)} targets")
        plt.grid(True); plt.legend()
        plt.savefig(os.path.join(experiment.dir, "training_curve_mean.png"), dpi=180)
        plt.close()

        # Per-target cam loss breakdown -- the thing to actually look at if
        # the shared model is trading one target off against another.
        plt.figure(figsize=(9, 5))
        for stem in target_stems:
            plt.plot(per_target_cam[stem], label=stem, linewidth=0.9)
        plt.xlabel("Epoch"); plt.ylabel("Cam Loss")
        plt.title(f"Batched SAIL [{run_config['physics_name']}] -- per-target cam loss")
        plt.grid(True); plt.legend(fontsize=6, ncol=3)
        plt.savefig(os.path.join(experiment.dir, "training_curve_per_target.png"), dpi=180)
        plt.close()

        np.savez(
            os.path.join(experiment.dir, "loss_curves.npz"),
            epoch_mean_total=np.array(epoch_mean_total),
            epoch_mean_cam=np.array(epoch_mean_cam),
            epoch_mean_sim=np.array(epoch_mean_sim),
            **{f"cam_{stem}": np.array(per_target_cam[stem]) for stem in target_stems},
        )

        t_run_elapsed_hr = (time.perf_counter() - t_run_start) / 3600
        print(f"\nRun config {run_config['physics_name']} complete: "
              f"{len(target_stems)} targets x {epochs} epochs in {t_run_elapsed_hr:.2f} hr.")
        run_summary.append({
            "physics_name": run_config["physics_name"],
            "targets": len(target_stems),
            "epochs": epochs,
            "best_mean_cam": best_mean_cam,
            "best_mean_epoch": best_mean_epoch,
            "elapsed_hr": t_run_elapsed_hr,
        })

    # ======================================================================
    # BOOKKEEPING
    # ======================================================================
    print(f"\n{'=' * 70}\nALL RUN CONFIGS COMPLETE ({len(run_summary)}/{len(run_configs)})\n{'=' * 70}")
    for rs in run_summary:
        print(f"  {rs['physics_name']:10s} -- {rs['targets']} targets x {rs['epochs']} epochs, "
              f"best mean cam {rs['best_mean_cam']:.6f} @ epoch {rs['best_mean_epoch']}, "
              f"{rs['elapsed_hr']:.2f} hr")
    total_hr = sum(rs["elapsed_hr"] for rs in run_summary)
    print(f"Total wall-clock across all run configs: {total_hr:.2f} hr")

finally:
    try:
        slm.close()
    except Exception as e:
        print("SLM close error (may already be closed):", e)
    try:
        camera.close()
    except Exception as e:
        print("Camera close error (may already be closed):", e)
    print("Closed camera and SLM.")

---

## Experimental Capture of Simulation-Generated Holograms

---

In [ ]:
"""
replay_simulation_holograms.py

Display every SIMULATION-generated hologram on the SLM and capture it, so the
simulation comparison (GS / GD / transformer / batched) has a matching
experimental column.

No training, no CITL, no optimisation. Each saved phase array is loaded,
pushed through the same phase_to_slm_frame pipeline the CITL scripts use,
displayed, and photographed once. The rig is identical to the CITL runs
(same ROI, rotation, DC settings, per-target exposure) so these captures are
directly comparable to everything else on the bench.

WHAT IT CAPTURES
----------------
  4 methods x 18 targets x 2 physics conditions = 144 captures, ~8 min of rig
  time. The capture KEY is decoupled from the on-disk SOURCE folder (see
  METHOD_SOURCES below), because the folder names do not say how a model was
  trained:

      capture key              reads from
      -----------------------  ------------------------------------------------
      gs                       simulation_comparison_{cond}/{stem}_*/outputs/gs/
                                   gs_iter_{N}_phase.npy
      gd                       ...                                  /outputs/gd/
                                   gd_iter_{N}_phase.npy
      transformer_per_target   ...                         /outputs/transformer/
                                   transformer_phase.npy
      transformer_batched      batched/1000px_P500_B18_{cond}_18imgs*/outputs/
                                   {stem}_best_phase.npy

  transformer_per_target and transformer_batched replace the older
  "transformer" / "shared_model" pair, which said nothing about the training
  regime and read as two unrelated methods rather than one architecture
  trained two ways.

OUTPUT LAYOUT -- chosen deliberately
------------------------------------
    {capture_root}/{condition}/{stem}/{key}.jpg

which is the shape build_dataset_manifest() in evaluate_methods.py expects
({capture_root}/{image_name}/{key}.jpg). Point evaluate_experimental_dataset()
at {capture_root}/{condition} and it scores these with no glue code, using the
same DC masking and metric as every other experimental result.

NOTE: build_dataset_manifest() has a HARDCODED key list. Add
"transformer_per_target" and "transformer_batched" to it, or these captures
will be silently skipped. Nothing else needs changing --
evaluate_single_target_experimental() routes any key outside its
`known_methods` set through the generic loop, which does identical work.

PRE-FLIGHT
----------
Every one of the 144 phase files is resolved and validated BEFORE the SLM or
camera is opened. A missing or ambiguous path fails immediately rather than
120 captures in -- rig time is the scarce resource here, and a half-finished
sweep is worse than one that never started.

Depends on: physics-free (no forward model needed -- these phases are already
final), stats_torch.phase_to_uint8, hardware.tile_to_slm_centered,
canon_camera, exposure_lookup, holoeye SDK.
"""

from datetime import datetime
import glob
import json
import os
import shutil
import time
from pathlib import Path

import numpy as np

from canon_camera import CanonEDSDKCamera
from holoeye import slmdisplaysdk

from utils import resolve_existing_path
from config_handler import ConfigHandler
from experiment_manager import ExperimentManager
from hardware import tile_to_slm_centered
from stats_torch import phase_to_uint8
from exposure_lookup import iso_to_hex, tv_to_hex


# ---------------------------------------------------------------------------
config_dir = resolve_existing_path(
    str(paths.CONFIGS / "citl"),
)
config_file = "simulation_capture"
config = ConfigHandler.load(config_file, search_paths=[config_dir])
source_config_path = os.path.join(config_dir, config_file + ".json")

capture_root = resolve_existing_path(*config.paths.capture_root, make=True)
sim_comparison_base = resolve_existing_path(*config.paths.simulation_comparison_base)
batched_base = resolve_existing_path(*config.paths.batched_simulation_base)
exposure_settings_path = resolve_existing_path(*config.paths.exposure_settings)

with open(exposure_settings_path) as f:
    exposure = json.load(f)

target_stems = list(exposure["targets"].keys())
subset = getattr(config.run, "target_subset", None)
if subset is not None:
    missing = [s for s in subset if s not in target_stems]
    if missing:
        raise ValueError(f"target_subset lists unknown targets: {missing}")
    target_stems = list(subset)

GS_ITERS = config.run.gs_iterations
GD_ITERS = config.run.gd_iterations
METHODS = list(config.run.methods)
CONDITIONS = list(config.run.conditions)

SLM_SHAPE = tuple(config.hardware.slm_shape)
SETTLE_TIME_S = config.hardware.settle_time_s
CAPTURE_TIMEOUT_S = config.hardware.capture_timeout_s
# Settle AFTER the capture, before the next exposure write. Without this the
# EDSDK returns EDS_ERR_DEVICE_BUSY (0x81) on the following set_iso/set_tv --
# the same failure that stopped batched SAIL, and this script sets exposure
# on every capture so it hits the identical pattern.
CAPTURE_SETTLE_S = getattr(config.hardware, "capture_settle_s", 0.5)
DLL_PATH = config.hardware.dll_path

m = config.hardware.manual_alignment
CAMERA_ROI = {"y0": m.y0, "x0": m.x0, "h": m.h, "w": m.w}
ROTATE_DEGREES = m.rotation_deg

H, W = config.run.height, config.run.width

print(f"targets    : {len(target_stems)}")
print(f"methods    : {METHODS}")
print(f"conditions : {CONDITIONS}")
print(f"GS @ {GS_ITERS} iters, GD @ {GD_ITERS} iters")
print(f"total captures: {len(target_stems) * len(METHODS) * len(CONDITIONS)}\n")


# ---------------------------------------------------------------------------
def _one(pattern, what):
    hits = sorted(glob.glob(pattern))
    if not hits:
        raise FileNotFoundError(f"{what}: nothing matches\n    {pattern}")
    if len(hits) > 1:
        raise RuntimeError(f"{what}: {len(hits)} matches, expected 1. Remove stale runs.\n"
                           + "\n".join(f"    {h}" for h in hits))
    return hits[0]


# Capture KEY -> (source, on-disk subdir, filename template).
#
# The key is what the capture is NAMED and what evaluate_methods.py looks for;
# the subdir is where the phase actually LIVES. These deliberately differ for
# the transformer: on disk it is still outputs/transformer/ (written by
# evaluate_transformer_once), but the capture is named transformer_per_target
# so it cannot be confused with the batched model. "shared_model" was the old
# name for that one and said nothing about how it was trained.
METHOD_SOURCES = {
    "gs":                     ("sim_comparison", "gs",          "gs_iter_{iters_gs}_phase.npy"),
    "gd":                     ("sim_comparison", "gd",          "gd_iter_{iters_gd}_phase.npy"),
    "transformer_per_target": ("sim_comparison", "transformer", "transformer_phase.npy"),
    "transformer_batched":    ("batched",        None,          "{stem}_best_phase.npy"),
}


def resolve_phase(method, stem, condition):
    """Locate one saved phase array. Raises rather than returning None -- a
    missing phase must stop the pre-flight, not silently drop a capture."""
    if method not in METHOD_SOURCES:
        raise ValueError(f"unknown method {method!r}. Choices: {list(METHOD_SOURCES)}")
    source, subdir, template = METHOD_SOURCES[method]
    fn = template.format(iters_gs=GS_ITERS, iters_gd=GD_ITERS, stem=stem)

    if source == "sim_comparison":
        run_dir = _one(
            os.path.join(sim_comparison_base, f"simulation_comparison_{condition}",
                         f"{stem}_simulation_comparison*"),
            f"{condition}/{stem} simulation_comparison folder")
        p = os.path.join(run_dir, "outputs", subdir, fn)
    else:
        run_dir = _one(
            os.path.join(batched_base, f"1000px_P500_B18_{condition}_18imgs*"),
            f"{condition} batched folder")
        p = os.path.join(run_dir, "outputs", fn)

    if not os.path.exists(p):
        raise FileNotFoundError(f"{condition}/{stem}/{method}: missing\n    {p}")
    return p


def phase_to_slm_frame(phase_np):
    """Identical to the CITL scripts: uint8 quantise -> horizontal flip ->
    centre on the SLM. Any deviation here would make these captures
    incomparable to the CITL ones."""
    phase_8bit = phase_to_uint8(np.asarray(phase_np, dtype=np.float32))
    holo_small = np.fliplr(phase_8bit).copy()
    holo_slm = tile_to_slm_centered(holo_small, slm_shape=SLM_SHAPE)
    return np.ascontiguousarray(holo_slm.astype(np.uint8))


# ---------------------------------------------------------------------------
# PRE-FLIGHT: resolve and validate everything before touching hardware.
# ---------------------------------------------------------------------------
print("=" * 74)
print("PRE-FLIGHT -- resolving every phase file before opening the rig")
print("=" * 74)

plan = []
errors = []
for condition in CONDITIONS:
    for stem in target_stems:
        for method in METHODS:
            try:
                p = resolve_phase(method, stem, condition)
                a = np.load(p, mmap_mode="r")
                if a.shape != (H, W):
                    errors.append(f"{condition}/{stem}/{method}: shape {a.shape}, expected ({H},{W})")
                    continue
                plan.append({"condition": condition, "stem": stem,
                             "method": method, "phase_path": p})
            except Exception as e:
                errors.append(str(e))

if errors:
    print(f"\n{len(errors)} PROBLEM(S) -- nothing was captured:\n")
    for e in errors:
        print(f"  {e}")
    raise SystemExit(1)

print(f"all {len(plan)} phase files resolved and shape-checked ({H}x{W})")
for condition in CONDITIONS:
    n = sum(1 for r in plan if r["condition"] == condition)
    print(f"  {condition:<10} {n:>4} captures")

# Exposure must exist for every target too -- check before the rig is open.
for stem in target_stems:
    e = exposure["targets"][stem]
    iso_to_hex(e["iso"] if e["iso"] is not None else exposure["default_iso"])
    tv_to_hex(e["tv"])
print("exposure settings resolve for every target\n")


# ---------------------------------------------------------------------------
experiment = ExperimentManager(name="replay_simulation", base_dir=capture_root,
                               overwrite=config.experiment.overwrite)
shutil.copy2(source_config_path, os.path.join(experiment.dir, "run_configuration.json"))
experiment.log(f"Date: {datetime.now()}")
experiment.log(f"Targets ({len(target_stems)}): {target_stems}")
experiment.log(f"Methods: {METHODS} | Conditions: {CONDITIONS}")
experiment.log(f"GS iters {GS_ITERS}, GD iters {GD_ITERS}")
experiment.log(f"Alignment: ROI x0={m.x0} y0={m.y0} w={m.w} h={m.h}, rot={m.rotation_deg}")

manifest = {"date": str(datetime.now()), "targets": target_stems,
            "methods": METHODS, "conditions": CONDITIONS,
            "gs_iterations": GS_ITERS, "gd_iterations": GD_ITERS,
            "capture_root": capture_root, "captures": []}

try:
    slm = slmdisplaysdk.SLMInstance()
    if not slm.requiresVersion(5):
        raise RuntimeError("Required SLM SDK version not available.")
    err = slm.open()
    if err != slmdisplaysdk.ErrorCode.NoError:
        raise RuntimeError(slm.errorString(err))
    print("SLM opened.")

    camera = CanonEDSDKCamera(
        dll_path=DLL_PATH,
        save_dir=os.path.join(experiment.dir, "_staging"),
        auto_set_save_to_host=True, auto_set_capacity=True, verbose=False)
    camera.initialize()
    camera.open_session()
    camera.assert_manual_mode()
    print("Camera opened, confirmed Manual (M) mode.\n")

    # Alignment is a fixed physical property of the rig; record it alongside
    # the captures so they can be processed later without guessing.
    with open(os.path.join(experiment.dir, "calibration.json"), "w") as f:
        json.dump({"mode": "manual", "roi": CAMERA_ROI,
                   "rotation_deg": ROTATE_DEGREES}, f, indent=2)

    t0 = time.perf_counter()
    for i, row in enumerate(plan, 1):
        cond, stem, method = row["condition"], row["stem"], row["method"]
        target_dir = os.path.join(experiment.dir, cond, stem)
        os.makedirs(target_dir, exist_ok=True)

        exp_entry = exposure["targets"][stem]
        iso_val = exp_entry["iso"] if exp_entry["iso"] is not None else exposure["default_iso"]
        tv_val = exp_entry["tv"]
        camera.set_iso(iso_to_hex(iso_val))
        camera.set_tv(tv_to_hex(tv_val))
        camera.save_dir = Path(target_dir)

        phase = np.load(row["phase_path"]).astype(np.float32)
        holo = phase_to_slm_frame(phase)

        err = slm.showData(holo)
        if err != slmdisplaysdk.ErrorCode.NoError:
            raise RuntimeError(f"SLM error on {cond}/{stem}/{method}: {slm.errorString(err)}")
        time.sleep(SETTLE_TIME_S)

        result = camera.capture_image(timeout_s=CAPTURE_TIMEOUT_S)
        time.sleep(CAPTURE_SETTLE_S)

        ext = os.path.splitext(result.path)[1] or ".jpg"
        dest = os.path.join(target_dir, f"{method}{ext}")
        if os.path.abspath(result.path) != os.path.abspath(dest):
            if os.path.exists(dest):
                os.remove(dest)
            shutil.move(result.path, dest)

        el = time.perf_counter() - t0
        eta = el / i * (len(plan) - i)
        print(f"[{i:>3}/{len(plan)}] {cond:<9} {stem:<14} {method:<13} "
              f"ISO {iso_val} Tv {tv_val:<7} -> {os.path.basename(dest)}  "
              f"({el/60:.1f} min, ~{eta/60:.1f} left)")
        experiment.log(f"{cond}/{stem}/{method} <- {row['phase_path']} "
                       f"| ISO={iso_val} Tv={tv_val} -> {dest}")

        manifest["captures"].append({**row, "capture_path": dest,
                                     "iso": iso_val, "tv": tv_val})
        with open(os.path.join(experiment.dir, "replay_manifest.json"), "w") as f:
            json.dump(manifest, f, indent=2)

    elapsed = time.perf_counter() - t0
    print(f"\n{len(plan)} captures in {elapsed/60:.1f} min")
    experiment.log(f"Complete: {len(plan)} captures in {elapsed/60:.1f} min")

finally:
    try:
        slm.close()
    except Exception as e:
        print("SLM close error (may already be closed):", e)
    try:
        camera.close()
    except Exception as e:
        print("Camera close error (may already be closed):", e)
    print("Closed camera and SLM.")

# Cosmetic only. A sync client can hold a handle on a freshly-emptied
# directory on Windows, and a failed tidy-up must never mask a successful run.
staging = os.path.join(experiment.dir, "_staging")
try:
    if os.path.isdir(staging) and not os.listdir(staging):
        os.rmdir(staging)
except OSError as e:
    print(f"(could not remove empty staging dir, harmless: {e})")

print(f"\nartefacts: {experiment.dir}")
print("\nTo score, point evaluate_experimental_dataset() at ONE condition at a time:")
for condition in CONDITIONS:
    print(f"    build_dataset_manifest(target_dir, r\"{os.path.join(experiment.dir, condition)}\")")

## Experimental Capture of Converged Holograms

Displays stored phase patterns on the SLM and photographs each one. No
training, no optimisation, no camera in the loop. Runs
`Scripts/replay_holograms.py` with
`configurations/.../citl/simulation_capture_converged.json`.

**What this run is for.** Two things at once. It answers the convergence
question, by putting GD at 10,000 iterations onto real optics for the first
time. And it puts *every* bench number in the paper onto one rig state, so no
comparison in the manuscript can be attributed to alignment drift between
sessions.

**Why it re-captures things already captured.** SAIL, batched SAIL and the CITL
arms were photographed in July. A converged-GD-versus-SAIL comparison assembled
from two alignments puts the alignment difference and the effect of interest
into the same number. The optical setup is confirmed unchanged today, so
everything is re-shot together and the whole comparison becomes internally
consistent.

**What it captures.** 14 methods, both physics conditions, all 18 targets, so
14 x 2 x 18 = 504 captures at roughly 3.3 s each, about 28 minutes of rig time.

| | |
|---|---|
| `gs_750`, `gs_10000` | GS at the published operating point and at convergence |
| `gd_750`, `gd_10000` | the same for GD. `gd_10000` under the faithful model is the strongest simulation-only baseline that exists for this rig |
| `transformer_per_target`, `transformer_batched` | one forward pass each, per-target and batched |
| `sail`, `sail_plus` | SAIL, and the dropped phase corrector, replayed from stored best phases |
| `batched_sail_750`, `batched_sail_2000` | batched SAIL at both epoch budgets |
| `gs_citl_{random,warm}`, `gd_citl_{random,warm}` | all four camera-in-the-loop arms |

`gs_750` and `gd_750` come from the same 10k sweep as their converged
counterparts, so the operating-point comparison sits inside one run and one
seed policy. `sail_plus` is captured despite being dropped from the manuscript,
because two minutes now is unrecoverable once the aberration sweep disturbs the
rig.

**Run it in two steps.** `--dry-run` resolves and shape-checks all 504 phase
files across five source trees and validates every exposure conversion, then
exits before the SLM or camera is opened. Only start the real run once it is
clean. Discovering a missing `best_phase.npy` 300 captures into a warm rig
wastes the session.

**Read the saturation column.** `gd_10000` concentrates more energy into the
signal region than `gd_750` and is photographed at an exposure metered for
`gd_750`. If it clips, its PSNR is wrong in the direction that flatters our
argument. Every capture reports the fraction of ROI pixels at or above 250, and
every warning is repeated at the end. If `gd_10000` clips where `gd_750` does
not, that target's comparison is void and needs re-metering.

**Do not delete the July run.** `experiments/replay_simulation/` stays. It is
the record behind the numbers already in front of reviewers, and the difference
between it and today on the shared methods is a free measurement of rig
reproducibility, which is worth reporting.

**Before scoring.** All 14 method keys are new to `build_dataset_manifest()` in
`evaluate_methods.py`, which has a hardcoded key list. Anything missing from it
is silently skipped.

In [ ]:
import os, sys, subprocess

SCRIPT = str(paths.ROOT / "code" / "method" / "replay_holograms.py")
CONFIG = "simulation_capture_converged"

# Step 1: pre-flight only, touches no hardware.
# ARGS = ["--config", CONFIG, "--dry-run"]

# Step 2: the real run. Comment out the line above and use this.
ARGS = ["--config", CONFIG]

# If it dies partway, continue into the same run directory:
# ARGS = ["--config", CONFIG, "--resume-dir",
#         r"...\experiments\replay_converged\replay_converged_YYYYMMDD_HHMMSS"]

# Narrow it while testing:
# ARGS = ["--config", CONFIG, "--dry-run", "--targets", "alley", "moon",
#         "--methods", "gd_750", "gd_10000", "--conditions", "ideal"]

# The subprocess inherits the kernel's import paths explicitly, so it resolves
# utils / config_handler / canon_camera the same way this notebook does.
env = {**os.environ, "PYTHONPATH": os.pathsep.join(p for p in sys.path if p)}

proc = subprocess.Popen([sys.executable, "-u", SCRIPT, *ARGS],
                        cwd=os.path.dirname(SCRIPT), env=env,
                        stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                        text=True, bufsize=1)
for line in proc.stdout:
    print(line, end="")
proc.wait()
print(f"\nexit code: {proc.returncode}")

In [ ]:
import numpy as np
import torch
from stats_torch import phase_to_field
from physics import hologram_intensity_from_field

path = str(next(paths.RESULTS.rglob("*best_phase*.npy")))  # any stored phase array in the deposit
phase = np.load(path)  # (H, W)
phase_t = torch.from_numpy(phase).unsqueeze(0)  # (1, H, W)

field = phase_to_field(phase_t)  # (1, 2, H, W) -- unit magnitude by construction (cos/sin)
I_pred, _ = hologram_intensity_from_field(
    field, return_field=True,
    pad_factor=1, apply_sinc=False, fill_factor=1, fill_is_areal=True,  # ideal, confirmed
)

I_np = I_pred[0].detach().cpu().numpy()
print("max:", I_np.max(), "mean:", I_np.mean(), "sum:", I_np.sum())
print("fraction of pixels above 1% of max:", (I_np > 0.01 * I_np.max()).mean())